https://github.com/raphaelmansuy/edgequake

https://www.youtube.com/watch?v=Cyn_Dm05_eU

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 📑 Table of Contents

---

### ⚙️ Phase 0: Infrastructure & Environment Setup
Establishing the project root, medallion directory structure, and audit trail.

### 📥 Phase 1: Ingestion, Validation & Silver Layer

* **1a** — Raw Ingestion & Structural Integrity Anchor
* **1b** — CDC ICD-10 Validation & Noisy 111 Audit (FY2026)
* **1c** — Token Volume & Truncation Audit
* **1d** — Raw Discovery Auditor *(human-in-the-loop)*
* **1e** — Pydantic Firewall & Silver Layer Registration
* **1f** — Surgical Signal Auditor *(human-in-the-loop)*
* **1g** — DuckDB Silver Vault Persistence

### 🔬 Phase 2: Gold Layer Construction

* **2a** — Biological Audit & Decimal Restoration
* **2b** — Gold Layer Discovery Auditor *(human-in-the-loop)*
* **2c** — ICD-10 Status Annotation

### 🔄 Phase 3: Signal Optimisation & Leakage Neutralisation

* **3a** — APSO-Flip (Diagnostic Signal Prioritisation)
* **3b** — ICD-10 Leakage Detection
* **3b.1** — Leakage Forensic Auditor *(human-in-the-loop)*
* **3c** — ICD-10 Code Redaction
* **3c.1** — Post-Redaction Integrity Auditor *(human-in-the-loop)*

### 💾 Phase 4: Gold Layer Parquet Export

---

### 📚 Dataset Reference
MedSynth (Rezaie Mianroodi et al., 2025) — arXiv:2508.01401

---

### 🎯 Project Objective

This notebook performs a rigorous forensic ingestion and preparation pipeline
for the MedSynth clinical dataset — 10,240 synthetic doctor-patient dialogue
and SOAP note pairs covering 2,037 distinct ICD-10 codes.

The pipeline moves from raw synthetic data through structural validation,
CDC reference verification, SOAP extraction, canonical code formatting,
ICD-10 status annotation, APSO-recomposition, and leakage redaction — producing
a fully annotated Gold layer ready for downstream ICD-10 classification or
clinical NLP tasks.

**Key findings:**
- 94.3% of records carry billable CDC FY2026 leaf codes
- 5.4% carry valid parent-level codes reflecting real clinical billing practice
- 64.7% of notes exceed ClinicalBERT's 512-token limit — addressed by APSO-Flip
- 28.5% of records contained explicit ICD-10 code strings — fully redacted
- All 10,240 records are retained; no records were removed

**Output:** `medsynth_gold_apso_20260319_111429.parquet` — 60.6 MB,
10,240 records, 13 columns, 0 ICD-10 leaks remaining.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🕵️ Project Narrative

This notebook transforms the raw MedSynth synthetic clinical dataset into a
fully validated, annotated, and redacted Gold layer suitable for downstream
ICD-10 classification and clinical NLP research.

The pipeline is structured as a progressive refinement across four stages:

**Ingestion & Validation (Phases 1a–1g)** — Raw data is loaded, structurally
validated, verified against the CDC FY2026 ICD-10 reference, and promoted
through Bronze → Silver layers with full audit traceability.

**Gold Layer Construction (Phases 2a–2c)** — ICD-10 codes are canonicalised
to decimal format, each record is annotated with a CDC-grounded status
classification, and the dataset is persisted as a queryable DuckDB vault.

**Signal Optimisation & Leakage Neutralisation (Phases 3a–3c)** — Clinical
notes are recomposed into APSO format to protect diagnostic signal from
truncation, explicit ICD-10 code strings are detected and redacted to prevent
label leakage, and both transformations are verified through human-in-the-loop
auditors and automated checks.

**Export (Phase 4)** — The Gold layer is persisted to a versioned,
schema-consistent Parquet artifact ready for downstream use.

Throughout, a Zero-Trust philosophy is applied: every assumption is validated
against evidence, every transformation is logged to an audit trail, and no
data is irreversibly discarded without documented justification.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📚 Dataset Reference: MedSynth (Rezaie Mianroodi et al., 2025)

> Rezaie Mianroodi, A., Rezaie, A., Todorov, N.G., Rakovski, C., & Rudzicz, F. (2025).
> *MedSynth: Realistic, Synthetic Medical Dialogue-Note Pairs.*
> arXiv:2508.01401v1 [cs.CL]
> Dataset: https://huggingface.co/datasets/Ahmad0067/MedSynth

MedSynth is a fully synthetic clinical dataset containing doctor-patient dialogue and
paired SOAP clinical notes, designed to advance automated medical documentation
research without HIPAA constraints.

---

### Dataset Composition

MedSynth contains **10,035 dialogue-note pairs** covering **2,001 unique ICD-10 codes**
at publication. Our loaded dataset of 10,240 records reflects the full HuggingFace
release. Key structural statistics from the paper:

| Field    | Avg Tokens | Avg Sentences |
|----------|------------|---------------|
| Dialogue | 932        | 55            |
| Note     | 621        | 23            |

These figures directly validate our Phase 1c token pressure findings: both fields
exceed the ClinicalBERT 512-token limit on average, confirming that truncation is the
structural default for this dataset, not an edge case.

---

### ICD-10 Code Distribution Strategy

The authors used **uniform sampling** — generating exactly 5 dialogue-note pairs per
code across the top 2,000 most frequent ICD-10 codes from the IQVIA PharMetrics Plus
database (800 million insurance claims) — despite real-world disease distributions
being highly skewed. This was intentional:

> *"We use this uniform sampling, despite the data skew, to prevent MedSynth from
> being dominated by common diseases."*

This directly explains our Phase 1g finding of **2,037 distinct ICD-10 codes with a
minimum frequency of 5**, and why Phase 2a found **0 rare labels (freq < 2)** —
equal representation is a design property, not a data quality outcome.

---

### ICD-10 Hierarchical Structure

ICD-10 codes follow a strict hierarchy that determines the granularity and billability
of each code — a key factor in the Noisy 111 analysis performed in Phase 1b:

- **Level 1 (Chapter):** `M` → Musculoskeletal diseases
- **Level 2 (Category):** `M25` → Other joint disorders *(non-billable parent)*
- **Level 3 (Subcategory):** `M25.5` → Pain in joint *(non-billable parent)*
- **Level 4 (Specificity):** `M25.56` → Pain in knee *(non-billable parent)*
- **Level 5 (Laterality):** `M25.562` → Pain in left knee ✅ *(billable leaf code)*

Only leaf-level codes are billable. Parent codes at levels 2–4 are valid ICD-10
identifiers but cannot be submitted for reimbursement — these form the **Noisy 111**
category identified in Phase 1b (5.4% of the dataset, 555 records). The presence of
parent codes in MedSynth reflects real clinical coding practice: the authors selected
from the top 2,001 most frequent codes in actual insurance claims, some of which are
inherently category-level.

---

### SOAP Structure

Notes are generated in strict SOAP format using a four-agent GPT-4o pipeline:
Scenario Provider → Scenario Judge → Note Writer → Note Polisher. This enforced
structure accounts for the **100% SOAP extraction success** achieved in Phase 1e.
The Subjective section optionally includes sub-sections (CC, HPI, ROS), which
explains the regex variation observed in the Phase 1f Surgical Signal Auditor.

---

### Intended Use & Limitations

MedSynth is designed for **model training and development**, not as a source of
clinical truth:

> *"MedSynth should not be considered a source of reliable medical information but
> rather a tool for model development."*

The dataset targets the **Dial-2-Note** (dialogue → clinical note) and **Note-2-Dial**
(clinical note → dialogue) tasks. Our pipeline's ICD-10 validation against the CDC
FY2026 reference is complementary to this intent — establishing label quality before
any downstream classification or fine-tuning work proceeds.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📋 Phase 1 Overview: The Anatomy of a MedSynth Record

Before ingestion begins, we establish a forensic baseline by mapping the
expected internal architecture of a MedSynth record. This map defines where
the diagnostic signal should be located and identifies where the pipeline is
most vulnerable to truncation and leakage.

### The Dual-Source Container

A single MedSynth record is a dual-source container, joining a structured
clinical summary with a raw doctor-patient transcript. Phase 1c quantifies
whether this combined text volume exceeds ClinicalBERT's 512-token context
window.
```mermaid
graph TD
    subgraph Record["MedSynth Dataset Record"]
        direction TB
        Note["📄 Clinical Note (Structured SOAP summary)"]
        Dialogue["💬 Doctor-Patient Dialogue (Raw transcript)"]
    end
```

### The SOAP Linear Progression

The Clinical Note follows the SOAP structure. In standard format, the
Assessment section — which contains the primary diagnostic signal — appears
third, after the Subjective and Objective sections. For long notes this means
the diagnostic signal is frequently truncated by a fixed context window.
The APSO-Flip in Phase 3a addresses this by reordering to place Assessment
first.
```mermaid
graph TD
    subgraph NoteStructure["Standard SOAP Ordering"]
        S["Subjective<br/>(History / Complaints)"]
        O["Objective<br/>(Exam / Vitals)"]
        A["Assessment<br/>(DIAGNOSTIC SIGNAL)"]
        P["Plan<br/>(Treatment / Next Steps)"]
        S --> O --> A --> P
    end
    style A fill:#f96,stroke:#333,stroke-width:4px
    NoteStructure -.-> Target((Truncation Risk))
```

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🛠️ Phase 0: Infrastructure & Environment Setup

Before ingestion begins, we must ensure the **project environment** is structurally
consistent and audit-ready. This cell establishes the foundational anchor that all
subsequent pipeline phases depend on [cite: 2026-02-10].

### 🛡️ Zero-Trust Infrastructure Protocol

1. **Project Root Alignment:** Absolute pathing is enforced by walking up the directory
   tree to locate `artifacts.yaml`. This ensures that all artifacts — logs, datasets,
   and outputs — are written to deterministic locations regardless of where the notebook
   is launched from [cite: 2026-02-10].

2. **Config & Registry Ingestion:** The `src.config` singleton is loaded and validated,
   confirming that the notebook's discovered root matches the config's declared root.
   This is the Single Source of Truth for all path resolution throughout the pipeline
   [cite: 2026-02-10].

3. **Medallion Directory Health Check:** All five layers of the medallion architecture
   are verified for existence and writability — `data/raw`, `data/silver`, `data/gold`,
   `db/`, and `outputs/evaluations`. Failures here surface I/O issues before any data
   is loaded [cite: 2026-01-29].

> **🛡️ Engineering Note:** Phase 0 is a prerequisite gate, not a transformation step.
> No data is loaded or modified here. Its sole purpose is to guarantee that every
> subsequent phase — ingestion, validation, EDA, and modelling — operates from a
> verified, portable, and fully traceable environment.

</div>

In [ ]:
# ==============================================================================
# PHASE 0: INFRASTRUCTURE & PATH ALIGNMENT (AUDIT READY)
# ==============================================================================

# Auto-reload src modules during development (IPython only)
%load_ext autoreload
%autoreload 2

import os
import sys
import json
from pathlib import Path
from datetime import datetime

# ==============================================================================
# 1. DISCOVER PROJECT ROOT (To enable 'import src')
# ==============================================================================
# Note: We do this minimal search ONLY to add root to sys.path. 
# Once config is loaded, config.project_root becomes the Single Source of Truth.
# ==============================================================================
print("🔍 Verifying Declarative Configuration...")

try:
    current = Path.cwd()
    while current != current.parent:
        if (current / "artifacts.yaml").exists():
            PROJECT_ROOT = current.resolve()
            break
        current = current.parent
    else:
        raise FileNotFoundError("Could not find artifacts.yaml in current or parent directories.")
    
    print(f"📍 Project Root discovered: .../{PROJECT_ROOT.name}")
    
except FileNotFoundError as e:
    print(f"❌ CRITICAL: Configuration file missing.")
    print(f"   Action: Ensure 'artifacts.yaml' exists in your project root.")
    raise e

# ==============================================================================
# 2. SYSTEM PATH INJECTION (Enable 'import src')
# ==============================================================================
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"📦 Project root added to sys.path")
else:
    print(f"📦 Project root already in sys.path")

# ==============================================================================
# 3. CONFIG & REGISTRY INGESTION (Single Source of Truth)
# ==============================================================================
try:
    from src.config import config
    
    # Production-safe validation (replaces unsafe 'assert' statements)
    required_attrs = ['project_root', 'log_path', 'log_event']
    missing = [attr for attr in required_attrs if not hasattr(config, attr)]
    if missing:
        raise RuntimeError(f"Config missing required attributes: {missing}. Verify src/config.py.")
    
    # Verify consistency between notebook discovery and config singleton
    if Path(config.project_root) != PROJECT_ROOT:
        raise RuntimeError(
            f"Config project_root mismatch!\n"
            f"  Notebook found: {PROJECT_ROOT}\n"
            f"  Config reports: {config.project_root}\n"
            f"  Action: Ensure only one artifacts.yaml exists in your hierarchy."
        )
    
    print(f"✅ Config loaded from: {config.config_path}")
    
except ImportError as e:
    print(f"❌ CRITICAL: Config import failed.")
    print(f"   Action: Verify src/config.py exists and dependencies are installed.")
    raise e
except RuntimeError as e:
    print(f"❌ CRITICAL: Config validation failed.")
    print(f"   Error: {e}")
    raise e

# ==============================================================================
# 4. SURGICAL ERA IMPORTS (With integrity verification)
# ==============================================================================
try:
    import polars as pl
    import duckdb
    
    # Defensive imports for Phase 1a (optional - will fail gracefully if missing)
    try:
        from src.data_loader import load_medsynth_raw
        print(f"   ✅ Data loader imported: load_medsynth_raw")
    except ImportError as e:
        print(f"   ⚠️  Data loader not available (will fail in Phase 1a): {e}")
        load_medsynth_raw = None
    
    print(f"✅ Core imports successful")
    print(f"   📊 Polars: {pl.__version__}")
    print(f"   🦆 DuckDB: {duckdb.__version__}")
    
except ImportError as e:
    print(f"❌ CRITICAL: Core library import failed.")
    print(f"   Action: Run 'pip install polars duckdb'")
    raise e

# ==============================================================================
# 5. MEDALLION DIRECTORY HEALTH CHECK (Using Config Methods)
# ==============================================================================
# Map logical names to config.resolve_path() keys for consistency
LAYER_PATH_KEYS = {
    "data/raw": ("data", "raw"),
    "data/silver": ("data", "silver"),
    "data/gold": ("data", "gold"),
    "db/": ("data", "db"),
    "outputs/evaluations": ("outputs", "evaluations"),
}

print("\n📁 Verifying Medallion Directory Structure...")

for layer_name, path_keys in LAYER_PATH_KEYS.items():
    try:
        # Use config resolver to ensure alignment with artifacts.yaml
        dir_path = config.resolve_path(*path_keys)
        dir_path.mkdir(parents=True, exist_ok=True)
        
        # Writability check
        test_file = dir_path / ".write_test"
        test_file.touch()
        test_file.unlink()
        print(f"   ✅ {layer_name}: writable")
        
    except KeyError:
        print(f"   ❌ {layer_name}: path keys not found in config")
        raise
    except (OSError, PermissionError) as e:
        print(f"   ❌ {layer_name}: not writable - {e}")
        raise

print(f"✅ Medallion layers verified (all writable)")

# ==============================================================================
# 6. AUDIT TRAIL INITIALIZATION (With Integrity Check)
# ==============================================================================
event_details = {
    "status": "Declarative configuration respected (no os.chdir)", 
    "root": str(PROJECT_ROOT),
    "config_path": str(config.config_path),
    "markers_found": [m for m in ["artifacts.yaml", "src"] 
                     if (PROJECT_ROOT / m).exists()],
    "cwd": str(Path.cwd()),
    # Note: timestamp added automatically by config.log_event()
}

config.log_event(
    phase="Phase 0: Environment Setup",
    action="initialize_session",
    details=event_details
)

# Verify log integrity: read last line to confirm event was written
if config.log_path.exists():
    try:
        with open(config.log_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
            last_entry = json.loads(lines[-1]) if lines else None
            
        if last_entry and last_entry.get("action") == "initialize_session":
            print(f"📝 Audit Trail Verified: {config.log_path}")
        else:
            print(f"⚠️  WARNING: Audit log exists but last entry mismatch")
    except (json.JSONDecodeError, IndexError) as e:
        print(f"⚠️  WARNING: Could not verify audit log content: {e}")
else:
    print(f"❌ ERROR: Audit log was not created at {config.log_path}")

# ==============================================================================
# 7. ZERO-TRUST VERDICT & BUSINESS SUMMARY
# ==============================================================================
print(f"\n✅ PHASE 0 COMPLETE: Environment secured for forensic audit (Zero-Trust)")

print(f"\n🛡️  ENVIRONMENT READINESS SUMMARY")
print(f"   " + "─" * 50)
print(f"   ✅ Project Root: {PROJECT_ROOT.name}")
print(f"   ✅ Data Layers: {len(LAYER_PATH_KEYS)} verified & writable")
print(f"   ✅ Audit Logging: Active")
print(f"   ✅ Reproducibility: Versions logged (Polars {pl.__version__}, DuckDB {duckdb.__version__})")
print(f"   " + "─" * 50)
print(f"   🎯 Business Impact: All data transformations will be")
print(f"      traceable, reproducible, and compliant with audit requirements.")

if Path.cwd() != PROJECT_ROOT:
    print(f"\n   ✅ Working directory preserved (Notebook portable)")
else:
    print(f"\n   ⚠️  Working directory matches project root (Acceptable)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## Phase 1a – Ingestion, Normalisation & Structural Integrity Anchor

This phase establishes a **reproducible raw anchor** for the MedSynth dataset, moving
the pipeline from an assumption-driven workflow to a **validated, typed environment**.

### The Five Pillars of Ingestion

1. **Polars-Native Ingestion**
   The dataset is loaded using `load_medsynth_raw()` into a strictly typed **Polars
   DataFrame**. If no primary key exists, a forensic `ID` column is generated to ensure
   every record can be uniquely traced throughout the pipeline. The dataset contains
   10,240 records — 205 more than the 10,035 reported at publication, consistent with
   post-publication additions to the HuggingFace release.

2. **ICD-10 Schema Normalisation**
   The source dataset stores ICD-10 codes as plain strings without decimal points.
   This convention reflects how codes are stored in the IQVIA PharMetrics Plus
   insurance claims database used by the MedSynth authors to define their code
   selection — the raw format is intentional, not a data quality issue. Codes are
   normalised here into a **`List[String]` structure** to support multi-label
   workflows and safe Polars list operations.

3. **ICD-10 Structural Normalisation (Raw Format Preserved)**
   Codes are validated for structural plausibility — correct length (3–8 characters),
   leading alpha character, and alphanumeric content. No decimal injection or
   canonicalisation is applied at this layer; the raw format is preserved exactly
   as ingested. The original string value is retained in `ICD10_raw_original` for
   full audit traceability. Billability validation against the FY2026 CDC reference
   is deferred to Phase 1b.

4. **DuckDB Raw Vault Registration**
   The validated frame is registered as the `raw_vault` table inside DuckDB,
   providing a high-performance SQL interface for auditing and downstream
   transformations without reloading from disk.

5. **Structural Integrity Audit (Zero-Trust Validation)**
   A SQL scan validates the dataset for missing notes, empty dialogues, and malformed
   label arrays. The audit flagged **2 empty dialogues** in this run — likely
   generation artifacts from the MedSynth pipeline rather than data corruption,
   given that the MedSynth Dialogue Polisher Agent was specifically designed to
   ensure all note content is reflected in the dialogue. These records are retained
   in the raw vault and given a formal disposition in Phase 1e via sentinel
   imputation (`[NO_TRANSCRIPT_AVAILABLE]`).

### Telemetry & Traceability

Audit metrics and ingestion metadata are logged to the central telemetry system,
capturing row counts, schema types, label coverage, and timestamped execution
details. This guarantees that every pipeline run produces a **traceable ingestion
record** for later auditing.

> **Result:** A typed Polars dataset, normalised ICD-10 labels in raw decimal-free
> format, a DuckDB SQL audit layer, and a persisted Parquet anchor — forming the
> **T-zero ingestion baseline** for all downstream pipeline stages. The 2 empty
> dialogue records are explicitly tracked forward to Phase 1e for documented
> disposition.

</div>

In [ ]:
# ==============================================================================
# PHASE 1a: INGESTION & THE PARQUET ANCHOR (TRULY IMMUTABLE RAW)
# ==============================================================================

import json
import re
import os
from pathlib import Path
from datetime import datetime
import polars as pl

# ==============================================================================
# HELPER FUNCTIONS (Validation Only - No Modification)
# ==============================================================================

def validate_icd10_format(code: str) -> bool:
    """
    Structural format check for ICD-10 codes stored WITHOUT decimals (raw source format).
    Accepts 3–8 alphanumeric characters starting with a letter.
    Note: Decimal presence is NOT required here — raw data omits decimals by convention.
    Full billability validation is deferred to Phase 1b (CDC FY2026 reference).
    """
    if not code or not isinstance(code, str):
        return False
    code = code.strip().upper()
    # Raw codes: letter + 2–7 alphanumeric chars (no decimal expected at this stage)
    pattern = r'^[A-Z][0-9A-Z]{2,7}$'
    return bool(re.match(pattern, code))

def split_icd10_codes(code_str: str) -> list[str]:
    """Split ICD-10 codes with awareness of potential multi-code formats."""
    if not code_str or not isinstance(code_str, str):
        return []

    parts = re.split(r'[;,|]\s*|\s+', code_str.strip())

    valid_codes = []
    for part in parts:
        part = part.strip().upper()
        if part and validate_icd10_format(part):
            valid_codes.append(part)
        elif part and not validate_icd10_format(part):
            potential_matches = re.findall(r'[A-Z][0-9A-Z]{2,7}', part)
            for pm in potential_matches:
                if validate_icd10_format(pm):
                    valid_codes.append(pm)

    return valid_codes if valid_codes else [p.strip().upper() for p in parts if p.strip()]

def to_native(value):
    """Convert any value to a native Python type."""
    if value is None:
        return None
    if hasattr(value, 'to_py'):
        return value.to_py()
    if hasattr(value, 'item'):
        return value.item()
    if isinstance(value, (int, float, str, bool)):
        return value
    try:
        return float(value)
    except (TypeError, ValueError):
        try:
            return str(value)
        except:
            return None

# ==============================================================================
# 1. INGEST VIA POLARS (NO DUMMY FALLBACK)
# ==============================================================================
print("📥 Ingesting raw MedSynth vault...")

try:
    from src.data_loader import load_medsynth_raw
    print(" ✅ Data loader imported successfully from src.data_loader")
except ImportError as e:
    raise ImportError(
        f"❌ CRITICAL: load_medsynth_raw not available.\n"
        f"   Please ensure src/data_loader.py exists and exports load_medsynth_raw.\n"
        f"   Error: {e}"
    ) from e

df_raw = load_medsynth_raw()

# ==============================================================================
# 1b. FORENSIC ID GENERATION
# ==============================================================================
if 'ID' not in df_raw.columns:
    print("⚠️ ID column missing. Generating unique forensic identifiers...")
    df_raw = df_raw.with_columns(
        pl.int_range(0, len(df_raw)).alias('ID')
    )
else:
    print("✅ ID column present in source data")

print(f" ✅ Loaded {len(df_raw):,} records")
print(f" 📊 Schema: {df_raw.schema}")

expected_cols = {'ID', 'Note', 'Dialogue', 'ICD10', 'ICD10_desc'}
actual_cols = set(df_raw.columns)
missing_cols = expected_cols - actual_cols

if missing_cols:
    print(f"⚠️ WARNING: Missing expected columns: {missing_cols}")
else:
    print(" ✅ All expected columns present")

# ==============================================================================
# 2. RAW DATA PREVIEW TABLE (ALL 5 CRITICAL FIELDS)
# ==============================================================================
print("\n" + "=" * 80)
print("📋 RAW DATA PREVIEW (First 10 Records - IMMUTABLE)")
print("=" * 80)
print("⚠️  Note: Text fields truncated for display readability.")
print("=" * 80)

preview_cols = ['ID', 'Note', 'Dialogue', 'ICD10', 'ICD10_desc']
missing_preview_cols = [col for col in preview_cols if col not in df_raw.columns]

if not missing_preview_cols:
    preview_df = df_raw.head(10).select([
        pl.col("ID"),
        pl.col("Note").str.slice(0, 100).alias("Note_preview"),
        pl.col("Dialogue").str.slice(0, 100).alias("Dialogue_preview"),
        pl.col("ICD10").alias("ICD10_raw"),
        pl.col("ICD10_desc").str.slice(0, 50).alias("ICD10_desc_preview")
    ])

    try:
        from IPython.display import display
        display(preview_df)
    except ImportError:
        print(preview_df)
else:
    print(f"⚠️  Missing columns for full preview: {missing_preview_cols}")

print("=" * 80)

# ==============================================================================
# 3. NORMALIZE ICD10 COLUMN (STRING → LIST[STR]) - NO DECIMAL INJECTION
# ==============================================================================
print("\n🔧 Normalizing ICD10 column to list[str]...")

# ✅ Preserve original raw value for audit
df_raw = df_raw.with_columns(
    pl.col("ICD10").alias("ICD10_raw_original")  # Immutable copy
)

if "ICD10" in df_raw.columns and df_raw.schema["ICD10"] == pl.Utf8:
    print(" 📝 Using enhanced ICD-10 code splitting...")
    df_raw = df_raw.with_columns(
        pl.col("ICD10")
        .map_elements(split_icd10_codes, return_dtype=pl.List(pl.String))
        .alias("ICD10")
    )

    sample_splits = df_raw.select(
        pl.col("ICD10").list.head(3)
    ).head(3).to_dicts()
    print(f" 📝 Sample splits: {sample_splits}")
elif "ICD10" in df_raw.columns and df_raw.schema["ICD10"] == pl.List(pl.String):
    print(" 📝 ICD10 column is already a list[str], skipping initial splitting.")
else:
    print(" ⚠️ ICD10 column not found or not in expected Utf8 format for splitting.")

print(f" 📊 ICD10 normalized type: {df_raw.schema['ICD10']}")

# ==============================================================================
# 4. STRUCTURAL SANITY CHECK (Format only — no billability claim)
# ==============================================================================
# NOTE: This section checks only that codes are structurally plausible
# (correct length, leading alpha character, alphanumeric content).
# Billability validation against the FY2026 CDC reference is intentionally
# deferred to Phase 1b, which downloads the official code set. Performing
# billability assessment here would be misleading because raw codes lack
# decimals by convention — a missing decimal is NOT a data quality issue
# at this layer.
# ==============================================================================
print("\n🔍 Structural Sanity Check on ICD-10 Codes (Raw State)...")
print("   ℹ️  Billability validation deferred to Phase 1b (CDC FY2026 reference).")

all_codes_df = df_raw.select(
    pl.col("ID"),
    pl.col("ICD10").list.explode().alias("code")
).filter(
    pl.col("code").is_not_null() & (pl.col("code").str.strip_chars() != "")
)

total_codes = all_codes_df.height

# Structural checks: plausible length (3–8 chars), starts with a letter, alphanumeric only
structural_results = all_codes_df.with_columns(
    pl.col("code").map_elements(
        lambda c: (
            isinstance(c, str)
            and 3 <= len(c.strip()) <= 8
            and c.strip()[0].isalpha()
            and c.strip().isalnum()
        ),
        return_dtype=pl.Boolean
    ).alias("structurally_valid")
)

valid_count   = structural_results.filter( pl.col("structurally_valid")).height
invalid_count = structural_results.filter(~pl.col("structurally_valid")).height
valid_pct     = (valid_count   / total_codes * 100) if total_codes > 0 else 0
invalid_pct   = (invalid_count / total_codes * 100) if total_codes > 0 else 0

print(f"\n📊 STRUCTURAL SANITY RESULTS:")
print(f"   Total ICD-10 codes:          {total_codes:,}")
print(f"   ✅ Structurally plausible:   {valid_count:,} ({valid_pct:.1f}%)")
print(f"   ⚠️  Structurally suspect:    {invalid_count:,} ({invalid_pct:.1f}%)")

if invalid_count > 0:
    print(f"\n⚠️  SAMPLE STRUCTURALLY SUSPECT CODES (First 10):")
    suspect_sample = structural_results.filter(
        ~pl.col("structurally_valid")
    ).select(["ID", "code"]).head(10)

    try:
        from IPython.display import display
        display(suspect_sample)
    except ImportError:
        print(suspect_sample)

print("=" * 80)

# ==============================================================================
# 5. REGISTER WITH DUCKDB (RAW LAYER)
# ==============================================================================
con = config.get_duckdb_conn()

try:
    con.register("raw_vault", df_raw)
    print("\n 🦆 Registered as 'raw_vault' in DuckDB")

    # Structural integrity check
    ghost_audit = con.execute("""
        WITH stats AS (
            SELECT
                COUNT(*) as total_records,
                SUM(CASE WHEN Note IS NULL OR TRIM(Note) = '' THEN 1 ELSE 0 END) as empty_notes,
                SUM(CASE WHEN Dialogue IS NULL OR TRIM(Dialogue) = '' THEN 1 ELSE 0 END) as empty_dialogues,
                SUM(CASE WHEN ICD10 IS NULL THEN 1 ELSE 0 END) as null_label_arrays,
                SUM(CASE WHEN ICD10 IS NOT NULL AND ARRAY_LENGTH(ICD10) = 0 THEN 1 ELSE 0 END) as empty_label_arrays,
                SUM(CASE WHEN ICD10 IS NOT NULL AND ARRAY_LENGTH(ICD10) > 0 THEN 1 ELSE 0 END) as records_with_labels
            FROM raw_vault
        )
        SELECT * FROM stats
    """).pl()

    print("\n🔍 Phase 1a: Structural Integrity Report")
    print("=" * 70)

    audit_dict = ghost_audit.to_dicts()[0]
    audit_dict_native = {k: to_native(v) for k, v in audit_dict.items()}

    main_metrics = ['total_records', 'empty_notes', 'empty_dialogues',
                    'null_label_arrays', 'empty_label_arrays', 'records_with_labels']
    for key in main_metrics:
        value = audit_dict_native.get(key, 0)
        print(f" {key:20s}: {int(value):>10,}")

    print("=" * 70)

    # ==============================================================================
    # 6. PERSIST RAW ANCHOR (IMMUTABLE)
    # ==============================================================================
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    raw_dir = config.resolve_path("data", "raw")
    anchor_path = raw_dir / f"raw_vault_anchor_{timestamp}.parquet"

    df_raw.write_parquet(anchor_path, compression="snappy")

    print(f"\n 💾 Raw anchor persisted: {anchor_path}")
    print(f"   ⚠️  This is IMMUTABLE - no ICD-10 modifications applied")

    # ==============================================================================
    # 7. LOG EVENT
    # ==============================================================================
    log_details = {
        "row_count": int(len(df_raw)),
        "anchor_path": str(anchor_path),
        "total_icd10_codes": total_codes,
        "structurally_valid_codes": valid_count,
        "structurally_valid_pct": round(valid_pct, 2),
        "structurally_suspect_codes": invalid_count,
        "structurally_suspect_pct": round(invalid_pct, 2),
        "billability_validation": "deferred_to_phase_1b",  # ✅ Explicit handoff
        "decimal_injection_applied": False,                 # ✅ Raw layer is immutable
        "empty_dialogue_count": int(audit_dict_native.get('empty_dialogues', 0)),
        "empty_note_count": int(audit_dict_native.get('empty_notes', 0)),
        "duckdb_connection_id": id(con)
    }

    config.log_event(
        phase="Phase 1a: Ingestion",
        action="raw_load_complete",
        details=log_details
    )

    print("\n📝 Audit trail updated")
    print("✅ Phase 1a complete: Raw vault anchored (IMMUTABLE)")

finally:
    try:
        con.close()
        print("\n🦆 DuckDB connection closed successfully.")
    except Exception as e:
        print(f"\n⚠️  Warning: Error closing DuckDB connection: {e}")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔍 Phase 1b: ICD-10 Clinical Validation & "Noisy 111" Audit
**Status:** Strategic Extraction & Validation (FY2026 CDC Standard)

This phase validates every ICD-10 code in the dataset against the official **FY2026
CDC Reference**, classifying each into one of three categories and establishing the
diagnostic signal profile before any downstream transformation is applied.

### 🛠️ Technical Implementation

The validation logic uses **Polars** for vectorised string processing and classifies
every code into one of three buckets:

1. **Billable (Leaf Codes):** Valid, specific codes present in the FY2026 CDC tabular
   reference. These represent the highest-quality diagnostic signals and are the
   primary targets for downstream classification.

2. **Non-Billable Parent (The "Noisy 111"):** Codes that exist in the ICD-10 hierarchy
   but are too broad for direct billing (e.g., `J18` rather than `J18.9`). These are
   not errors — the MedSynth authors selected codes from the top 2,001 most frequent
   ICD-10 codes in real IQVIA insurance claims data, and clinicians do legitimately
   submit category-level codes in practice. The 5.4% Noisy 111 rate reflects real
   coding behaviour in the source claims data, not a generation artifact.

3. **Absent from CDC Reference:** Codes not found in the FY2026 tabular descriptions
   file. The 25 codes in this category (e.g., `T781XXA`, `H4011X1`) use the ICD-10-CM
   placeholder `X` convention for injury/trauma codes — a structural feature of the
   coding system where character positions must be held for specificity extensions.
   Their absence from the CM descriptions file is a known gap in the tabular reference,
   not a dataset error. These codes are confirmed as valid ICD-10-CM codes by the
   MedSynth paper's methodology, which used IQVIA claims filtered to ICD-10 codes
   applied after 2015.

### 📊 Validation Objectives

* **Quantify Signal Profile:** Measure the proportion of billable, parent-level, and
  reference-absent codes to establish the diagnostic label quality baseline.
* **Audit Readiness:** Confirm every code against the CDC source-of-truth, moving
  from assumption to validated schema.
* **Surface Reference Gaps:** Identify codes absent from the tabular reference for
  documented disposition before any modelling phase.

### 📊 Observed Results

* **94.3% billable** — strong signal density consistent with the paper's design intent
* **5.4% Noisy 111** — 555 records carrying category-level parent codes; top codes
  at 10 occurrences each (`E784`, `R972`, `R312`, `Z9889`, `R938`)
* **0.24% absent from CDC reference** — 25 placeholder-X injury codes; retained
  in the raw vault with disposition documented in Phase 1e

> **Note on Noisy 111:** These records are not discarded at this layer. The presence
> of parent-level codes reflects real clinical coding practice in the IQVIA source
> data. Their narratives will be assessed in downstream phases to determine whether
> sufficient clinical specificity exists to support classification tasks.

</div>

In [ ]:
# ==============================================================================
# Phase 1b: ICD-10 VALIDATION & NOISY 111 AUDIT (FY2026 CDC)
# ==============================================================================

import requests
import zipfile
import io
import polars as pl

# ------------------------------------------------------------------------------
# 1. ROBUST CDC DATA LOAD
# ------------------------------------------------------------------------------
def load_cdc_icd10_2026():
    url = "https://ftp.cdc.gov/pub/health_statistics/nchs/publications/ICD10CM/2026/icd10cm-Code%20Descriptions-2026.zip"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # Guard against CDC changing internal zip structure between annual releases
        target_file = next(
            (f for f in z.namelist() if 'codes' in f.lower() and f.endswith('.txt')),
            None
        )
        if target_file is None:
            raise FileNotFoundError(
                f"Could not find a 'codes*.txt' file in CDC zip. "
                f"Available files: {z.namelist()}"
            )

        with z.open(target_file) as f:
            lines = [line.decode('utf-8', errors='ignore').strip() for line in f]

    df = pl.DataFrame({"raw": lines}).filter(pl.col("raw").str.len_chars() > 0)

    df = df.with_columns([
        pl.col("raw").str.extract(r"^\s*(\S+)", 1)
          .str.replace(".", "", literal=True)
          .str.to_uppercase()
          .alias("code_no_decimal"),
        pl.col("raw").str.extract(r"^\s*\S+\s+(.+)$", 1).alias("description")
    ])

    return df.select(["code_no_decimal", "description"]).unique()

# ------------------------------------------------------------------------------
# 2. LOAD REFERENCE DATA
# ------------------------------------------------------------------------------
cdc_df = load_cdc_icd10_2026()
print(f"✅ CDC Reference Loaded: {cdc_df.height:,} billable codes found.")

# ------------------------------------------------------------------------------
# 3. PREPARE DATASET (df_raw)
# ------------------------------------------------------------------------------
df_codes = df_raw.select([
    pl.col("ID"),
    pl.col("ICD10").list.explode().alias("code")
]).with_columns([
    # Raw codes have no decimals at this stage (Phase 1a stores them without,
    # reflecting the IQVIA claims database convention used by the MedSynth authors).
    # This replace() is a defensive guard for any edge cases that slip through.
    pl.col("code")
    .str.replace(".", "", literal=True)
    .str.to_uppercase()
    .str.strip_chars()
    .alias("code_no_decimal")
])

# ------------------------------------------------------------------------------
# 4. PARENT (NON-BILLABLE) LOOKUP TABLE
# ------------------------------------------------------------------------------
# Build all valid prefixes from billable leaf codes to identify parent/category codes.
# range(3, len(c)) intentionally excludes the code itself and any code shorter
# than 4 chars, since those cannot have children in the ICD-10 hierarchy.
#
# KNOWN LIMITATION: This approach identifies a code as a parent only if it is a
# prefix of at least one billable leaf code in the CDC FY2026 reference. A
# 3-character code that has NO billable children in the reference (e.g. an
# obsolete or highly specialised category) would fall into 'invalid_or_malformed'
# rather than 'non_billable_parent'. Given that MedSynth codes were selected from
# the top 2,001 most frequent ICD-10 codes in real insurance claims — all of which
# have active billing descendants — this edge case is unlikely to affect results
# materially, but is documented here for audit transparency.
all_billable = set(cdc_df.get_column("code_no_decimal").to_list())
parent_prefixes = set()
for c in all_billable:
    for i in range(3, len(c)):
        parent_prefixes.add(c[:i])

parent_df = pl.DataFrame({
    "code_no_decimal": list(parent_prefixes),
    "is_parent": [True] * len(parent_prefixes)
})

# ------------------------------------------------------------------------------
# 5. EXECUTE VALIDATION JOINS
# ------------------------------------------------------------------------------
df_checked = (
    df_codes
    .join(cdc_df, on="code_no_decimal", how="left")
    .join(parent_df, on="code_no_decimal", how="left")
    .with_columns([
        pl.col("is_parent").fill_null(False)
    ])
)

# ------------------------------------------------------------------------------
# 6. FINAL CLASSIFICATION LOGIC
# ------------------------------------------------------------------------------
df_checked = df_checked.with_columns([
    pl.when(pl.col("description").is_not_null())
    .then(pl.lit("billable"))
    .when(pl.col("is_parent"))
    .then(pl.lit("non_billable_parent (Noisy 111)"))
    .otherwise(pl.lit("invalid_or_malformed"))
    .alias("status")
])

# ------------------------------------------------------------------------------
# 7. SUMMARY REPORTS
# ------------------------------------------------------------------------------
summary = df_checked.group_by("status").agg(pl.len().alias("count")).with_columns([
    (pl.col("count") / pl.col("count").sum() * 100).alias("pct")
])

print("\n📊 FINAL VALIDATION SUMMARY:")
print(summary)

if not df_checked.filter(pl.col("status") == "non_billable_parent (Noisy 111)").is_empty():
    print("\n🚨 NOISY 111 FREQUENCY (Top 10):")
    noisy_report = (
        df_checked.filter(pl.col("status") == "non_billable_parent (Noisy 111)")
        .group_by("code")
        .agg(pl.len().alias("occurrences"))
        .sort("occurrences", descending=True)
    )
    print(noisy_report.head(10))

# ------------------------------------------------------------------------------
# 8. DEBUG REMAINING INVALIDS
# ------------------------------------------------------------------------------
invalids = df_checked.filter(pl.col("status") == "invalid_or_malformed")
if not invalids.is_empty():
    print(f"\n🔍 DEBUG: {invalids.height} codes remain 'invalid':")
    print(invalids.select(["code", "code_no_decimal"]).unique().head(10))

# ------------------------------------------------------------------------------
# 9. AUDIT TRAIL
# ------------------------------------------------------------------------------
noisy_count   = df_checked.filter(pl.col("status") == "non_billable_parent (Noisy 111)").height
invalid_count = invalids.height

config.log_event(
    phase="Phase 1b: ICD-10 Validation",
    action="cdc_validation_complete",
    details={
        "cdc_reference_codes": cdc_df.height,
        "total_codes_checked": df_checked.height,
        "validation_summary": summary.to_dicts(),
        "noisy_111_count": noisy_count,
        "invalid_count": invalid_count,
    }
)

print("\n📝 Audit trail updated")
print("✅ Phase 1b complete: ICD-10 codes validated against FY2026 CDC reference.")

## ICD10 description 

In [ ]:
# Quick summary of unique categories
unique_label_descriptions = df_raw["ICD10_desc"].n_unique()

unique_labels = df_raw["ICD10"].n_unique()


print(f"Unique ICD-10 Labels {unique_labels}")
print(f"Unique ICD-10 Label descriptions {unique_label_descriptions}")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 1c: Token Volume & Truncation Audit

Having confirmed structural integrity and ICD-10 validity in Phases 1a and 1b, we
now quantify the spatial constraints imposed by transformer-based models. This is a
prerequisite step before any architectural decisions are made about text representation.

### ⚖️ The Tokenomics Heuristic

Transformer models process tokens, not raw words. We apply a standard heuristic for
medical narratives to estimate token counts without running a full tokeniser pass:

$$Estimated\ Tokens = Words \times 1.3$$

This estimate is applied to both text fields — **Notes** (structured SOAP narratives)
and **Dialogues** (raw doctor-patient transcripts) — and plotted against the
**ClinicalBERT 512-token context limit**.

**Important caveat:** This heuristic was calibrated against general medical text.
ClinicalBERT uses WordPiece tokenisation, which tends to produce more tokens per
word than the BPE tokenisers used in LLaMA and Mistral — the models used to report
token counts in the MedSynth paper (932 avg for Dialogues, 621 avg for Notes). The
`words × 1.3` multiplier is therefore likely a *conservative* estimate of truncation
risk for ClinicalBERT specifically. The true truncation rates are at least as high
as reported here, and may be higher.

### 🕵️ Audit Objectives

* **Identify the Blind Zone:** Visualise the proportion of Notes and Dialogues that
  exceed the 512-token limit, beyond which the model receives no signal.
* **Quantify Source Density:** Compare token distributions across both text sources
  to understand which carries more truncation risk.
* **Validate Against Source Paper:** Cross-reference empirical token distributions
  against the MedSynth paper's reported averages to confirm the heuristic is
  producing plausible estimates.

### 📊 Observed Results

| Field | Paper Avg (tokens) | Empirical Risk (> 512) |
|---|---|---|
| Note | 621 | 64.7% |
| Dialogue | 932 | 100.0% |

Both fields exceed the 512-token limit on average — consistent with the paper's own
reported statistics. The Note distribution peaks just above the truncation cliff at
approximately 520 tokens, meaning the majority of notes lose their tail content under
a standard 512-token window. The Dialogue distribution sits entirely to the right of
the cliff, peaking around 950 tokens, confirming that dialogue truncation is total
and unavoidable without architectural intervention.

### 🛡️ Implications for Text Architecture (APSO-Flip)

Standard SOAP notes place the **Assessment & Plan** at the end of the document. For
a model with a fixed 512-token context window, content beyond that limit is invisible.
This means the primary diagnostic signal — the coded diagnosis embedded in the
Assessment section — is routinely truncated away before the model ever processes it.

The **APSO-Flip** transformation addresses this by reordering the note structure to
place the **Assessment** first, ensuring the highest-signal content survives truncation
regardless of document length. Given that the MedSynth Note Writer Agent enforced
strict SOAP ordering during generation, and that our Phase 1e SOAP extraction achieved
100% success across all four sections, the Assessment content is reliably isolatable
for this reordering.

> **🛡️ Zero-Trust Verdict:** With 64.7% of Notes and 100% of Dialogues exceeding
> the context window, truncation is not an edge case — it is the structural default
> for this dataset, confirmed by both our empirical measurement and the MedSynth
> paper's own reported token averages. APSO-Flip transformation is required before
> any text field is passed to a token-limited model.

</div>

In [ ]:
# ==============================================================================
# PHASE 1c: TOKEN VOLUME & TRUNCATION AUDIT
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

TOKEN_HEURISTIC = 1.3
BERT_LIMIT = 512

# ==============================================================================
# 1. CALCULATE WORD COUNTS (Whitespace-Safe Regex)
# ==============================================================================
print("📊 Calculating token pressure metrics...")

df_lengths = df_raw.select([
    pl.col("Note")
      .str.count_matches(r"\S+")
      .fill_null(0)
      .alias("note_words"),

    pl.col("Dialogue")
      .str.count_matches(r"\S+")
      .fill_null(0)
      .alias("dialogue_words")
])

# ==============================================================================
# 2. CONVERT TO PANDAS FOR VISUALIZATION
# ==============================================================================
pdf = df_lengths.to_pandas()

# ==============================================================================
# 3. APPLY TOKEN HEURISTIC
# ==============================================================================
pdf["note_tokens_est"]     = pdf["note_words"]     * TOKEN_HEURISTIC
pdf["dialogue_tokens_est"] = pdf["dialogue_words"] * TOKEN_HEURISTIC

# ==============================================================================
# 4. COMPUTE TRUNCATION RISK METRICS
# ==============================================================================
note_at_risk     = (pdf["note_tokens_est"]     > BERT_LIMIT).sum()
dialogue_at_risk = (pdf["dialogue_tokens_est"] > BERT_LIMIT).sum()

note_at_risk_pct     = (note_at_risk     / len(pdf) * 100) if len(pdf) > 0 else 0
dialogue_at_risk_pct = (dialogue_at_risk / len(pdf) * 100) if len(pdf) > 0 else 0

print(f"   📈 Note Truncation Risk:     {note_at_risk_pct:.1f}%")
print(f"   📈 Dialogue Truncation Risk: {dialogue_at_risk_pct:.1f}%")

# ==============================================================================
# 5. VISUALIZATION SETUP
# ==============================================================================
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

# Sample if >10k rows for performance
display_size = min(len(pdf), 5000)
if len(pdf) > display_size:
    print(f"   ⚠️  Sampling {display_size} rows for visualization (total: {len(pdf)})")
    pdf_sample = pdf.sample(n=display_size, random_state=42)
else:
    pdf_sample = pdf

sns.kdeplot(
    pdf_sample["note_tokens_est"],
    fill=True,
    label="Note (Est. Tokens)",
    color="#2980b9",
    alpha=0.5
)

sns.kdeplot(
    pdf_sample["dialogue_tokens_est"],
    fill=True,
    label="Dialogue (Est. Tokens)",
    color="#27ae60",
    alpha=0.5
)

# ClinicalBERT truncation wall
plt.axvline(
    x=BERT_LIMIT,
    color="#c0392b",
    linestyle="--",
    linewidth=3,
    label="ClinicalBERT Limit (512)"
)

# Chart formatting
plt.title(
    "Forensic Analysis: Token Pressure vs. Context Window",
    fontsize=14,
    fontweight="bold"
)
plt.xlabel("Estimated Tokens (Words × 1.3)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.xlim(0, 1500)
plt.legend()

plt.show()
plt.close()  # Prevent memory leaks from matplotlib figures

# ==============================================================================
# 6. VISUAL AUDIT SUMMARY
# ==============================================================================
print("\n📊 VISUAL AUDIT COMPLETE")
print(f"   ⚠️  {note_at_risk_pct:.1f}% of Notes will be truncated without APSO-Flip optimisation.")
print(f"   ⚠️  {dialogue_at_risk_pct:.1f}% of Dialogues exceed the context window.")

# ==============================================================================
# 7. LOG THE TOKEN AUDIT FOR TRACEABILITY
# ==============================================================================
config.log_event(
    phase="Phase 1c: Token Audit",
    action="token_pressure_analysis_complete",
    details={
        "total_records": len(pdf),
        "note_truncation_risk_pct": round(note_at_risk_pct, 2),
        "dialogue_truncation_risk_pct": round(dialogue_at_risk_pct, 2),
        "bert_context_limit": BERT_LIMIT,
        "token_heuristic_multiplier": TOKEN_HEURISTIC,
        "audit_note": (
            f"{note_at_risk_pct:.1f}% of notes and "
            f"{dialogue_at_risk_pct:.1f}% of dialogues exceed "
            f"ClinicalBERT {BERT_LIMIT}-token limit"
        )
    }
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 1c complete: Token pressure quantified and logged")

# ==============================================================================
# 8. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")

if note_at_risk_pct > 50:
    print(f"   ⚠️  {note_at_risk_pct:.1f}% of Notes exceed context window.")
    print(f"   ⚠️  {dialogue_at_risk_pct:.1f}% of Dialogues exceed context window.")
    print(f"   ✅ APSO-Flip transformation is REQUIRED to protect diagnostic signal.")
else:
    print(f"   ✅ {note_at_risk_pct:.1f}% of Notes within context window.")
    print(f"   ⚠️  {dialogue_at_risk_pct:.1f}% of Dialogues exceed context window.")
    print(f"   ℹ️  Verify dialogue handling strategy before downstream modelling.")

# ==============================================================================
# 9. BUSINESS-FACING SUMMARY
# ==============================================================================
print(f"\n💼 BUSINESS IMPACT SUMMARY")
print(f"   " + "─" * 50)
print(f"   📊 Total Records Analyzed:      {len(pdf):,}")
print(f"   ⚠️  Notes at Risk:              {note_at_risk_pct:.1f}%")
print(f"   ⚠️  Dialogues at Risk:          {dialogue_at_risk_pct:.1f}%")
print(f"   🎯 Recommended Action:          APSO-Flip Transformation")
print(f"   " + "─" * 50)
print(f"   💡 Without optimisation, ~{note_at_risk_pct:.0f}% of Notes and")
print(f"      ~{dialogue_at_risk_pct:.0f}% of Dialogues will lose diagnostic")
print(f"      signal beyond the {BERT_LIMIT}-token truncation cliff.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🕵️ Phase 1d: Raw MedSynth Discovery Auditor

Having completed automated structural validation (Phase 1a), CDC billability
validation (Phase 1b), and token pressure analysis (Phase 1c), we now perform a
**manual discovery audit** of the raw dataset. This browser-native interface allows
direct observation of the raw text fields before any transformation logic is applied.

This is a human-in-the-loop step. Its findings inform downstream design decisions
but are not captured in the automated audit trail.

### Discovery Objectives

1. **Direct Observation:** A high-fidelity browser view of the three core fields —
   **Note**, **Dialogue**, and **ICD-10 Label with description** — bypassing
   notebook rendering constraints for full-length text inspection.

2. **Narrative Correlation:** Visual audit of the relationship between the clinical
   Note and its corresponding Dialogue. In MedSynth, Notes are the primary generated
   artifact — produced first by the Note Writer and Note Polisher Agents — and
   Dialogues are derived from them by the Dialogue Generator and Dialogue Polisher
   Agents. This auditor confirms that the derivation is coherent and that diagnostic
   signal present in the Note is faithfully reflected in the Dialogue.

3. **Noisy 111 Spot-Check:** Manual inspection of records carrying non-billable
   parent codes (identified in Phase 1b). These codes reflect real clinical coding
   practice in the IQVIA source data. The objective here is to observe the narrative
   depth and clinical specificity of these records — understanding whether the
   associated notes contain sufficient detail for downstream classification tasks
   despite the broad code assignment.

4. **SOAP Structure Baseline:** Direct observation of the raw `Note` format
   establishes the baseline for SOAP-section extraction patterns required in
   subsequent phases. Combined with the token pressure findings from Phase 1c —
   where 64.7% of Notes exceed the 512-token limit — this informs the APSO-Flip
   transformation strategy: reordering note sections to place the Assessment first,
   ensuring the primary diagnostic signal survives truncation.

> **Result:** A ground-truth view of the raw dataset providing the observational
> foundation for extraction pattern design and structured mapping in subsequent
> phases. Any patterns or anomalies observed here should be documented as inline
> notes before proceeding to Phase 1e.

</div>

In [ ]:
# ==============================================================================
# PHASE 1d: RAW DISCOVERY AUDITOR (INK-BLACK HIGH CONTRAST)
# ==============================================================================

# ==============================================================================
# DEPENDENCY CHECK
# ==============================================================================
try:
    import panel as pn
    import warnings
    import time
except ImportError as e:
    missing_pkg = str(e).split("'")[1] if "'" in str(e) else "panel"
    raise ImportError(
        f"❌ Missing dependency: {missing_pkg}\n"
        f"   Install with: pip install {missing_pkg}\n"
        f"   Or: pip install panel matplotlib seaborn (for all Phase 1 deps)"
    ) from e

# Ensure Panel extensions are loaded
pn.extension(design='material')


def launch_browser_auditor(df_pl, start_index=51):
    """
    Launches a dedicated browser tab to audit raw MedSynth data.
    Enforces Ink-Black rendering for high-resolution displays.
    Displays both ICD-10 code and description side-by-side.

    Parameters
    ----------
    df_pl : polars.DataFrame
        The raw DataFrame with 'Note', 'Dialogue', 'ICD10', and 'ICD10_desc' columns.
    start_index : int, optional
        Initial index to display.

    Returns
    -------
    threading.Thread
        The StoppableThread running the server (can be used to stop).
    """
    if df_pl is None or len(df_pl) == 0:
        raise ValueError("DataFrame is empty or None – cannot launch auditor.")

    # Validate required columns exist
    required_cols = ['Note', 'Dialogue', 'ICD10']
    missing_cols = [col for col in required_cols if col not in df_pl.columns]

    if missing_cols:
        raise ValueError(
            f"❌ Missing required columns for auditor: {missing_cols}\n"
            f"   Available columns: {df_pl.columns}\n"
            f"   Please run Phase 1a first to ensure proper data structure."
        )

    # Memory efficiency warning
    if len(df_pl) > 100000:
        print(f"⚠️  Large dataset detected ({len(df_pl):,} rows). Converting to pandas...")
        print(f"   Memory usage will temporarily increase.")

    # Convert to pandas once for efficient iloc access
    df_pd = df_pl.to_pandas()
    max_idx = len(df_pd) - 1

    # Slider for index navigation
    index_slider = pn.widgets.IntSlider(
        name='Examine Record (Index)',
        start=0, end=max_idx, value=min(start_index, max_idx),
        bar_color='#2196f3', sizing_mode='stretch_width'
    )

    # Stop button
    stop_button = pn.widgets.Button(name='🛑 Stop Auditor', button_type='danger', disabled=True)

    # CSS to enforce Ink-Black, Bold Monospace, and Auto-Wrap
    raw_text_style = """
    <style>
        .ink-black-audit {
            white-space: pre-wrap !important;
            word-wrap: break-word !important;
            font-family: 'Courier New', Courier, monospace !important;
            font-size: 1.0rem !important;
            font-weight: 700 !important;
            color: #000000 !important;
            background-color: #ffffff !important;
            padding: 20px;
            border: 2px solid #000;
            border-radius: 4px;
            line-height: 1.5;
            -webkit-font-smoothing: none;
        }
        .code-desc-container {
            display: flex;
            flex-direction: row;
            gap: 20px;
            margin: 20px 0;
            width: 100%;
            max-width: 850px;
        }
        .code-box {
            flex: 1;
            background-color: #e3f2fd;
            border: 3px solid #1976d2;
            border-radius: 10px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }
        .desc-box {
            flex: 2;
            background-color: #f3e5f5;
            border: 3px solid #7b1fa2;
            border-radius: 10px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }
        .code-label {
            font-size: 0.9rem;
            color: #0d47a1;
            font-weight: bold;
            text-transform: uppercase;
            letter-spacing: 1px;
            display: block;
            margin-bottom: 10px;
        }
        .desc-label {
            font-size: 0.9rem;
            color: #4a148c;
            font-weight: bold;
            text-transform: uppercase;
            letter-spacing: 1px;
            display: block;
            margin-bottom: 10px;
        }
        .code-value {
            font-size: 2.5rem;
            color: #0d47a1;
            font-weight: 900;
            font-family: 'Courier New', monospace;
            line-height: 1.2;
        }
        .desc-value {
            font-size: 1.5rem;
            color: #4a148c;
            font-weight: 600;
            line-height: 1.3;
        }
    </style>
    """

    def format_raw(text):
        """Wrap raw text in the ink-black styling."""
        if text is None:
            text = "N/A"
        return f'{raw_text_style}<div class="ink-black-audit">{text}</div>'

    def clean_label(label) -> str:
        """
        Safely convert an ICD-10 label to a clean display string.
        Handles: Python list, tuple, numpy array, and stringified list
        representations that result from Polars List→pandas conversion.
        """
        if isinstance(label, (list, tuple)):
            return ', '.join(str(c) for c in label)
        if isinstance(label, str) and label.startswith('['):
            # Handles pandas stringifying a Polars List column as "['E669']"
            return label.strip('[]').replace("'", '').replace('"', '').strip()
        # Fallback: numpy scalar or plain string
        try:
            import numpy as np
            if isinstance(label, np.ndarray):
                return ', '.join(str(c) for c in label.tolist())
        except ImportError:
            pass
        return str(label) if label is not None else 'N/A'

    @pn.depends(index_slider)
    def get_raw_view(index):
        row = df_pd.iloc[index]
        note     = row.get('Note',      'N/A')
        dialogue = row.get('Dialogue',  'N/A')
        label    = row.get('ICD10',     'N/A')
        desc     = row.get('ICD10_desc','No description available')

        label_str = clean_label(label)

        # Side-by-side HTML for code and description
        code_desc_html = f"""
        {raw_text_style}
        <div class="code-desc-container">
            <div class="code-box">
                <span class="code-label">ICD-10 CODE</span>
                <span class="code-value">{label_str}</span>
            </div>
            <div class="desc-box">
                <span class="desc-label">DESCRIPTION</span>
                <span class="desc-value">{desc}</span>
            </div>
        </div>
        """

        return pn.Column(
            pn.pane.HTML(code_desc_html, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 📝 Raw Note Content",
                    pn.pane.HTML(format_raw(note), sizing_mode='stretch_width'),
                    height=750, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Raw Dialogue Content",
                    pn.pane.HTML(format_raw(dialogue), sizing_mode='stretch_width'),
                    height=750, scroll=True, sizing_mode='stretch_width'
                ),
                sizing_mode='stretch_width'
            ),
            sizing_mode='stretch_width'
        )

    # Build the application layout
    header = pn.pane.Markdown(
        "# 🕵️ MedSynth Raw Discovery Auditor\n"
        "**Browser Mode:** Ink-Black rendering enabled for maximum contrast.\n\n"
        "### 📖 How to Use:\n"
        "- **Slider**: Navigate through records by index\n"
        "- **Left Panel**: Clinical Note (SOAP format)\n"
        "- **Right Panel**: Doctor-Patient Dialogue\n"
        "- **Top Boxes**: ICD-10 Code + Description\n"
        "- **Stop Button**: Shut down the auditor server",
        sizing_mode='stretch_width'
    )

    app = pn.Column(
        header,
        pn.Row(index_slider, stop_button),
        get_raw_view,
        sizing_mode='stretch_width'
    )

    # Launch the server
    server_thread = pn.serve(
        app,
        show=True,
        threaded=True,
        port=0,
        title="MedSynth Raw Auditor"
    )

    # Enable stop button after server starts
    stop_button.disabled = False

    def stop_server(event):
        try:
            server_thread.stop()
            app[:] = [pn.pane.Markdown("### 🛑 Auditor stopped. You may close this tab.")]
            stop_button.disabled = True
            print("✅ Auditor server stopped successfully.")
        except Exception as e:
            warnings.warn(f"Error stopping server: {e}")
            print(f"⚠️  Error stopping server: {e}")

    stop_button.on_click(stop_server)

    # Wait for server to be ready
    timeout = 5
    start_time = time.time()
    while not hasattr(server_thread, 'server') or server_thread.server is None:
        if time.time() - start_time > timeout:
            print("⚠️  Server did not become ready within timeout.")
            break
        time.sleep(0.1)

    if hasattr(server_thread, 'server') and server_thread.server is not None:
        actual_server = server_thread.server
        app_url = getattr(actual_server, 'app_url', None)
        if app_url:
            print(f"✅ Auditor launched at {app_url}")
            print(f"   Use the Stop button to shut down the server.")
        else:
            print(f"✅ Auditor launched – check your browser for the new tab.")
    else:
        print(f"✅ Auditor launched – check your browser for the new tab.")

    return server_thread


# ==============================================================================
# LAUNCH AUDITOR
# ==============================================================================
if 'df_raw' in locals() and df_raw is not None:
    print("🚀 Launching MedSynth Raw Discovery Auditor (Phase 1d)...")

    # Ensure ICD10_desc exists — add placeholder if missing
    if 'ICD10_desc' not in df_raw.columns:
        print("⚠️  ICD10_desc column not found - adding placeholder")
        df_raw = df_raw.with_columns([
            pl.lit("Description not available").alias("ICD10_desc")
        ])

    try:
        auditor_server = launch_browser_auditor(df_raw, start_index=51)
        print("\n💡 TIP: Use the slider to examine different records.")
        print("   Look for: SOAP structure, code-description alignment, and narrative quality.")
        print("   Pay particular attention to records flagged as Noisy 111 in Phase 1b.")
    except Exception as e:
        print(f"❌ Failed to launch auditor: {e}")
        raise
else:
    print("❌ df_raw not found. Please run Phase 1a first.")
    print("   The Raw Vault must be loaded before auditing.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🛡️ Phase 1e: Pydantic Firewall & Silver Layer Registration

Having validated ICD-10 codes against the CDC reference (Phase 1b) and quantified
token pressure (Phase 1c), we now enforce structural schema validation across every
record before promoting the dataset to the **Silver layer**. No record enters the
analytical pipeline without passing this gate.

---

### 🛡️ The Gatekeeper: What is being validated?

The **Pydantic Gatekeeper** (`src/gatekeeper.py`) enforces the following schema
against every record:

* **ID Integrity:** Every record must carry a unique, non-null string anchor for
  downstream audit traceability.

* **Narrative Presence:** The `Note` and `Dialogue` fields must contain text signal.
  Records with null values are imputed with explicit sentinel placeholders
  (`[NO_TRANSCRIPT_AVAILABLE]`, `[EMPTY_NOTE_SIGNAL]`) rather than silently dropped,
  preserving record count while flagging null sections for downstream handlers.
  The 2 empty dialogues identified in Phase 1a are resolved here — retained in the
  Silver layer with their sentinel values intact, with disposition logged to the
  audit trail.

* **SOAP Section Extraction:** The gatekeeper splits each `Note` into its four
  structural components — **Subjective, Objective, Assessment, and Plan** — using
  regex patterns. Successful isolation of the Assessment section is the critical
  prerequisite for the APSO-Flip transformation in Phase 2.

* **Signal Density Check:** The extraction rate of the Assessment section is reported
  explicitly. A rate below 80% would indicate that regex patterns require adjustment
  before proceeding.

### The Surgical Strategy

1. **Deterministic ID Injection:** Where no primary key exists, a row index is cast
   to a string-typed `ID` column, acting as a permanent forensic anchor across all
   subsequent transformations.

2. **Sentinel Imputation:** Null fields are replaced with explicit placeholders rather
   than dropped, ensuring the record count remains stable and null sections are
   identifiable by downstream tokenisers.

3. **Silver Artifact Registration:** All validated records are registered via
   `config.register_dataframe`, creating a persistent recovery point before any
   further transformation is applied.

### On the 100% SOAP Extraction Rate

The SOAP extraction depth report confirms 100% success across all four sections.
This result is expected given the dataset's synthetic origin — the MedSynth Note
Writer Agent was explicitly instructed to produce notes in strict SOAP format, and
the Note Polisher Agent was designed to enforce correct section placement before
the notes were finalised. The structural consistency of the notes is a feature of
controlled synthetic generation, not a property that should be assumed to generalise
to real clinical notes from EHR systems.

The optional sub-sections within Subjective (Chief Complaint, History of Present
Illness, Review of Systems) vary stochastically across records — the Note Writer
Agent prompt instructed the model to "roll a dice" to determine whether to include
them. This explains the regex variation observed in the Phase 1f Surgical Signal
Auditor.

> **Result:** A schema-validated Silver dataset where all 10,240 records have passed
> structural inspection, all four SOAP sections have been successfully isolated, and
> the 2 empty dialogue records have been formally dispositioned via sentinel
> imputation — forming a clean, traceable baseline for all downstream transformation
> phases.

</div>

In [ ]:
# ==============================================================================
# PHASE 1e: PYDANTIC FIREWALL & SILVER LAYER REGISTRATION
# ==============================================================================

import polars as pl
from datetime import datetime

# ==============================================================================
# 1. IMPORT GATEKEEPER
# ==============================================================================
try:
    from src.gatekeeper import validate_dataframe, ClinicalRecord
    print("✅ Successfully imported validate_dataframe and ClinicalRecord from src/gatekeeper.py")
except ImportError as e:
    print(f"❌ CRITICAL: Failed to import from src.gatekeeper.")
    print(f"   Please ensure src/gatekeeper.py exists and Phase 0 setup is correct.")
    print(f"   Error details: {e}")
    raise

# ==============================================================================
# 2. ID CASTING & SENTINEL IMPUTATION
# ==============================================================================
# Note: The 2 empty dialogues identified in Phase 1a are resolved here.
# They are retained via sentinel imputation rather than dropped, preserving
# record count and making null sections explicitly identifiable downstream.
# ==============================================================================
if 'ID' not in df_raw.columns:
    print("⚠️ ID column missing. Generating unique forensic identifiers...")
    df_prepared = df_raw.with_row_index("ID").with_columns([
        pl.col("ID").cast(pl.String),
        pl.col("Dialogue").fill_null("[NO_TRANSCRIPT_AVAILABLE]"),
        pl.col("Note").fill_null("[EMPTY_NOTE_SIGNAL]")
    ])
else:
    print("✅ ID column already exists. Casting to String type...")
    df_prepared = df_raw.with_columns([
        pl.col("ID").cast(pl.String),
        pl.col("Dialogue").fill_null("[NO_TRANSCRIPT_AVAILABLE]"),
        pl.col("Note").fill_null("[EMPTY_NOTE_SIGNAL]")
    ])

# Confirm sentinel imputation resolved the known empty dialogue records
imputed_dialogues = (df_prepared["Dialogue"] == "[NO_TRANSCRIPT_AVAILABLE]").sum()
imputed_notes     = (df_prepared["Note"]     == "[EMPTY_NOTE_SIGNAL]").sum()
if imputed_dialogues > 0 or imputed_notes > 0:
    print(f"   ℹ️  Sentinel imputation applied:")
    print(f"      Dialogues imputed: {imputed_dialogues} (expected: 2 from Phase 1a)")
    print(f"      Notes imputed:     {imputed_notes}")
else:
    print("   ✅ No null fields requiring imputation.")

# ==============================================================================
# 3. EXECUTE VALIDATION WITH SCHEMA DISCLOSURE
# ==============================================================================
print("\n🛡️ Initializing Zero-Trust Validation Firewall...")
print(" ↳ Enforcing ID Unique Anchors")
print(" ↳ Isolating SOAP Clinical Sections")
print(" ↳ Verifying Diagnostic Signal Density")

df_valid_polars, df_errors, df_valid_objects = validate_dataframe(df_prepared)
df_valid = df_valid_polars

# ==============================================================================
# 4. REGISTER THE SILVER DATASET
# ==============================================================================
config.register_dataframe("silver_medsynth", df_valid, phase="Phase 1e")

# ==============================================================================
# 5. FORENSIC REPORT & TELEMETRY
# ==============================================================================
print(f"\n✅ VALIDATION COMPLETE")
print(f"-------------------------------------------")
print(f"✅ Valid Records:        {len(df_valid):,} (Promoted to Silver)")
print(f"⚠️  Quarantined:         {len(df_errors):,} (Isolated for Review)")
print(f"🧬 SOAP Objects Extracted: {len(df_valid_objects):,} (Successfully)")

if len(df_errors) > 0:
    print("\n❌ Error Forensic Sample (first 5):")
    if not df_errors.is_empty():
        for row in df_errors.head(5).iter_rows(return_type="dict"):
            print(f"   ID: {row.get('ID', 'N/A')}, Error: {row.get('error_message', 'N/A')}")
    else:
        print("   No errors in sample.")

# ==============================================================================
# 5b. SOAP EXTRACTION DEPTH REPORT
# ==============================================================================
print(f"\n🧬 SOAP EXTRACTION DEPTH REPORT")
print(f"   " + "─" * 50)

soap_sections = ['subjective', 'objective', 'assessment', 'plan']
for section in soap_sections:
    extracted_count = sum(
        1 for obj in df_valid_objects
        if getattr(obj, section, None) is not None
    )
    extracted_pct = (
        extracted_count / len(df_valid_objects) * 100
    ) if len(df_valid_objects) > 0 else 0
    print(f"   {section.capitalize():12s}: {extracted_count:,}/{len(df_valid_objects):,} ({extracted_pct:.1f}%)")

print(f"   " + "─" * 50)

assessment_count = sum(1 for obj in df_valid_objects if obj.assessment is not None)
assessment_pct = (
    assessment_count / len(df_valid_objects) * 100
) if len(df_valid_objects) > 0 else 0

if assessment_pct < 80:
    print(f"   ⚠️  WARNING: Only {assessment_pct:.1f}% of records have Assessment extracted.")
    print(f"      Regex patterns may need adjustment before Phase 2 (APSO-Flip).")
else:
    print(f"   ✅ Assessment extraction rate: {assessment_pct:.1f}% (Sufficient for APSO-Flip)")

# ==============================================================================
# 5c. BUSINESS-FACING SUMMARY
# ==============================================================================
print(f"\n💼 BUSINESS IMPACT SUMMARY")
print(f"   " + "─" * 50)
print(f"   📊 Total Records Processed:    {len(df_prepared):,}")
print(f"   ✅ Validated for Modelling:    {len(df_valid):,} ({len(df_valid)/len(df_prepared)*100:.1f}%)")
print(f"   ❌ Quarantined for Review:     {len(df_errors):,} ({len(df_errors)/len(df_prepared)*100:.1f}%)")
print(f"   🧬 SOAP Sections Extracted:    {len(df_valid_objects):,}")
print(f"   ℹ️  Sentinel Records (Dialogue):{imputed_dialogues:,} (retained with placeholder)")
print(f"   " + "─" * 50)
print(f"   🎯 Silver layer is ready for APSO-Flip transformation (Phase 2).")

# ==============================================================================
# 6. LOG THE EVENT
# ==============================================================================
config.log_event(
    phase="Phase 1e: Pydantic Firewall",
    action="pydantic_firewall_complete",
    details={
        "valid_count": len(df_valid),
        "error_count": len(df_errors),
        "object_count": len(df_valid_objects),
        "assessment_extraction_pct": round(assessment_pct, 2),
        "imputed_dialogues": int(imputed_dialogues),
        "imputed_notes": int(imputed_notes),
        "empty_dialogue_disposition": "retained_with_sentinel"
    }
)

print(f"\n📝 Audit trail updated")
print(f"✅ PHASE 1e COMPLETE: Silver dataset registered and validated")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🧪 Phase 1f: Surgical Signal Auditor

Following the Pydantic Firewall in Phase 1e, we launch a browser-native auditor
to visually verify that SOAP section extraction was performed correctly across
the Silver dataset. This is a human-in-the-loop confirmation step before any
further transformation is applied.

### Audit Objectives

1. **Visual Verification:** A side-by-side Panel layout compares the raw Clinical
   Note (unparsed, left panel) against the extracted SOAP Signal (structured,
   right panel) produced by the Pydantic Gatekeeper's regex logic. The raw note
   is already in SOAP format — this auditor confirms that the regex extraction
   has correctly identified and isolated each section boundary.

2. **Section Integrity:** Each of the four SOAP sections is rendered in a distinct
   clinical colour — Subjective (slate), Objective (teal), Assessment (medical red),
   Plan (blue) — making extraction gaps or boundary misalignments immediately
   visible. Priority inspection targets the **Assessment** section (medical red),
   which carries the primary diagnostic signal and is the target of the APSO-Flip
   transformation in Phase 2.

3. **Noisy 111 Spot-Check:** Records carrying non-billable parent codes (identified
   in Phase 1b) can be navigated to directly using the slider and their known index
   positions. The objective is to observe the narrative depth and clinical specificity
   of these records — understanding how the MedSynth generation pipeline handled
   broad category-level diagnoses, and whether the SOAP extraction captures the
   relevant clinical content regardless of code granularity.

4. **Traceability Anchor:** The auditor cross-references records via the deterministic
   string `ID` injected in Phase 1e, maintaining a transparent audit chain from raw
   source to structured Silver layer.

> **Result:** A web-based Surgical Auditor confirming extraction quality before
> proceeding to Canonical Label Mapping in Phase 2. Any extraction boundary failures
> or section misalignments observed here should be documented as inline notes before
> continuing.

</div>

In [ ]:
# ==============================================================================
# PHASE 1f: SURGICAL SIGNAL AUDITOR (BROWSER-NATIVE)
# ==============================================================================

import panel as pn

pn.extension(design='material')


def launch_signal_auditor(df_prepared, validated_list):
    """
    Launches a dedicated browser tab to compare Raw Notes vs. Extracted SOAP
    signals produced by the Phase 1e Pydantic Gatekeeper.

    Parameters
    ----------
    df_prepared : polars.DataFrame
        The prepared DataFrame from Phase 1e (with string-cast ID column).
    validated_list : list[ClinicalRecord]
        The list of validated Pydantic objects from Phase 1e.
    """
    # 1. Map validated objects by ID for instant cross-referencing
    val_map = {str(v.id): v for v in validated_list}
    df_pd = df_prepared.to_pandas()

    # 2. Slider — defaults to record 51 as a representative mid-dataset sample
    index_slider = pn.widgets.IntSlider(
        name='Examine Extraction Quality (Index)',
        start=0, end=len(df_pd) - 1, value=51,
        bar_color='#e91e63', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def update_comparison(index):
        row = df_pd.iloc[index]
        row_id = str(row['ID'])
        validated_obj = val_map.get(row_id)

        if not validated_obj:
            return pn.pane.Alert(
                f"⚠️ No validated signal for ID: {row_id}",
                alert_type="warning"
            )

        # Clinical colour palette — consistent across all SOAP sections
        colors = {
            'subjective': '#607d8b',   # Slate
            'objective':  '#009688',   # Teal
            'assessment': '#d32f2f',   # Medical Red — highest diagnostic signal
            'plan':       '#1976d2'    # Blue
        }

        sections = {
            'subjective': validated_obj.subjective,
            'objective':  validated_obj.objective,
            'assessment': validated_obj.assessment,
            'plan':       validated_obj.plan
        }

        extracted_html = '<div style="font-family: sans-serif; line-height: 1.6;">'
        for title, content in sections.items():
            color = colors.get(title, '#555')
            if content:
                extracted_html += f"""
                <div style="margin-bottom: 15px; padding: 12px; border-radius: 6px;
                            background: #fff; border: 1px solid #eee;
                            border-top: 5px solid {color};">
                    <span style="font-size: 0.7rem; font-weight: 900;
                                 color: {color}; text-transform: uppercase;">{title}</span>
                    <div style="font-size: 0.9rem; color: #333;
                                white-space: pre-wrap;">{content.strip()}</div>
                </div>"""
        extracted_html += "</div>"

        return pn.Row(
            pn.Column(
                "### 🔴 Raw Clinical Note",
                pn.pane.Str(row['Note']),
                height=750, scroll=True, sizing_mode='stretch_width'
            ),
            pn.Column(
                "### 🟢 Extracted SOAP Signal",
                pn.pane.HTML(extracted_html),
                height=750, scroll=True, sizing_mode='stretch_width'
            )
        )

    # Dashboard layout
    layout = pn.Column(
        "# 🧪 Surgical Extraction Audit",
        "**Objective:** Confirm the Phase 1e Pydantic Gatekeeper isolated SOAP "
        "sections correctly before Canonical Label Mapping (Phase 2).",
        index_slider,
        update_comparison,
        sizing_mode='stretch_width'
    )

    return layout.show(title="MedSynth Signal Auditor — Phase 1f", threaded=True)


# ==============================================================================
# LAUNCH THE AUDITOR
# Uses df_prepared and df_valid_objects produced by Phase 1e
# ==============================================================================
if 'df_prepared' not in locals() or 'df_valid_objects' not in locals():
    print("❌ df_prepared or df_valid_objects not found.")
    print("   Please run Phase 1e (Pydantic Firewall) before launching this auditor.")
else:
    print("🚀 Launching Surgical Signal Auditor (Phase 1f)...")
    print("   💡 TIP: Use the slider to navigate records.")
    print("   💡 TIP: Pay particular attention to Assessment sections (Medical Red).")
    print("   💡 TIP: Cross-check Noisy 111 codes from Phase 1b using their known indices.")
    signal_server = launch_signal_auditor(df_prepared, df_valid_objects)

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🦆 Phase 1g: DuckDB Persistence & Silver Vault

With `df_valid` secured by the Phase 1e Pydantic Firewall, we now transition from
volatile in-memory storage to a persistent **Silver Vault** within DuckDB. This
ensures the validated dataset remains queryable via SQL across kernel restarts and
provides the foundation for all downstream SQL-based transformations.

### Persistence Strategy

1. **Table Materialisation:** The validated Polars DataFrame is promoted to a
   permanent `silver_vault` table in DuckDB. Unlike a temporary view, this table
   acts as the Silver layer anchor — a high-performance, persistent source for all
   subsequent clinical transformations.

2. **Schema Verification:** A `DESCRIBE` operation confirms that all column types
   have been correctly mapped from Polars to DuckDB, including the string-cast `id`
   column and the `VARCHAR[]` array type for the `label` column.

3. **Label Cardinality Audit:** A SQL scan using `UNNEST` counts distinct individual
   ICD-10 code strings across all label arrays, establishing the code density
   baseline before any Gold layer transformation is applied.

   Note: `COUNT(DISTINCT label)` on a `VARCHAR[]` column in DuckDB produces
   unreliable results due to array hashing behaviour — `UNNEST` is required to
   count distinct individual code strings correctly. The confirmed count of
   **2,037 distinct ICD-10 codes** is consistent with the MedSynth paper's
   reported 2,001 unique codes, with the difference attributable to post-publication
   additions to the HuggingFace release.

4. **State Transition Logging:** The vault initialisation is captured in the project
   telemetry, recording the table structure and unique code count at the moment of
   persistence.

> **Result:** A persistent, SQL-verifiable Silver Vault — 10,240 validated records
> across 2,037 distinct ICD-10 codes — providing the foundation for decimal
> restoration and Gold layer promotion in Phase 2a.

</div>

In [ ]:
# ==============================================================================
# PHASE 1g: DUCKDB PERSISTENCE (THE SILVER VAULT)
# ==============================================================================

# 'config' object is available globally from Phase 0.
con = config.get_duckdb_conn()

try:
    # Register the validated Polars DataFrame with this DuckDB connection so it
    # can be referenced by name in subsequent SQL statements.
    con.register("df_valid", df_valid)

    # 1. Materialise as a permanent Silver Vault table
    con.execute("CREATE OR REPLACE TABLE silver_vault AS SELECT * FROM df_valid")

    # 2. Verify schema — confirm column types mapped correctly from Polars to DuckDB
    db_audit = con.execute("DESCRIBE silver_vault").pl()
    print("📊 DuckDB Silver Vault Schema (Validated):")
    print(db_audit)

    # 3. Label Cardinality Audit
    #
    # IMPORTANT: COUNT(DISTINCT label) on a VARCHAR[] (array) column in DuckDB
    # does not reliably count distinct array values — it hashes or serialises
    # arrays in a way that produces spurious uniqueness (e.g. 2,037 instead of
    # the true ~73). This is a known DuckDB behaviour with list-type columns.
    #
    # The correct approach is to UNNEST the label arrays into individual code
    # strings first, then count distinct values across those strings.
    # COUNT(DISTINCT id) is used for total_records to remain correct even if
    # multi-code records are ever introduced.
    label_stats = con.execute("""
        SELECT
            COUNT(DISTINCT code) as unique_codes,
            COUNT(DISTINCT id)   as total_records
        FROM (
            SELECT id, UNNEST(label) as code
            FROM silver_vault
        )
    """).pl()

    print("\n📈 Label Cardinality Report:")
    print(label_stats)

    # Sanity check — confirm total_records matches expected row count
    raw_count = con.execute("SELECT COUNT(*) as n FROM silver_vault").pl()["n"].item()
    if label_stats["total_records"].item() != raw_count:
        print(f"\n⚠️  WARNING: Cardinality record count ({label_stats['total_records'].item():,}) "
              f"does not match raw vault count ({raw_count:,}).")
        print(f"   This may indicate multi-code records in the dataset.")
    else:
        print(f"\n   ✅ Record count consistent: {raw_count:,} records")

    # 4. Log state transition
    config.log_event(
        phase="Phase 1g: Persistence",
        action="duckdb_vault_initialized",
        details={
            "table": "silver_vault",
            "unique_codes": label_stats["unique_codes"].item(),
            "total_records": label_stats["total_records"].item(),
            "count_method": "UNNEST — counts distinct individual ICD-10 code strings"
        }
    )

    print("\n✅ PHASE 1g COMPLETE: Silver vault initialized and logged.")

finally:
    con.close()
    print("🦆 DuckDB connection for Phase 1g closed.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 2a: Biological Audit & Decimal Restoration

With the Silver Vault persisted in Phase 1g, we now apply the first transformation
to the validated dataset, promoting it to the **Gold layer**. This phase addresses
two things: confirming the label frequency distribution and canonicalising ICD-10
code formatting.

### Surgical Standardisation

1. **Forensic Label Frequency Analysis:** The DuckDB `silver_vault` is queried to
   profile label frequency across all 2,037 distinct ICD-10 codes. This serves as
   a sanity check against the MedSynth paper's stated design: uniform sampling of
   exactly 5 dialogue-note pairs per code. We expect zero rare labels (freq < 2)
   and a minimum frequency of 5 across all codes. Any deviation would indicate
   either a data loading issue or an undocumented change to the HuggingFace release.

2. **Canonical Decimal Restoration:** Raw MedSynth codes omit the ICD-10 decimal
   point (e.g. `M25562`). We restore the canonical format by inserting a decimal
   after the third character (e.g. `M25562 → M25.562`), matching the CDC FY2026
   standard. Note that Phase 1b validation was performed in decimal-free space, so
   this is a canonicalisation step only — it does not alter billability
   classifications.

   3-character parent codes (e.g. `R32`, `J18`) are a special case — they have no
   subclassification component so no decimal is inserted, leaving them as-is. This
   matches their representation in the CDC FY2026 reference.

3. **Gold Layer Promotion:** The standardised records are registered as the
   `gold_medsynth` artifact, creating a recovery point that carries both the raw
   label array and the canonical `standard_icd10` string.

4. **Audit Visibility:** The `standard_icd10` column is displayed immediately after
   restoration to provide visual confirmation that the decimal insertion logic is
   functioning correctly before any downstream transformation consumes it.

> **Result:** A standardised Gold dataset with canonical ICD-10 codes and a confirmed
> uniform label distribution — 2,037 distinct codes, minimum frequency 5, zero rare
> labels — consistent with the MedSynth paper's uniform sampling design.

</div>

In [ ]:
# ==============================================================================
# PHASE 2a: BIOLOGICAL AUDIT & DECIMAL RESTORATION (GOLD LAYER)
# ==============================================================================

import polars as pl
from datetime import datetime

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 2a Prerequisites...")

if 'df_valid' not in locals():
    print("❌ CRITICAL: df_valid not found. Please run Phase 1e first.")
    raise ValueError("df_valid not available in local scope")

print(f"   ✅ df_valid available: {len(df_valid):,} records")

# ==============================================================================
# 2. OPEN FRESH DUCKDB CONNECTION
# ==============================================================================
# Phase 1g closed its connection on completion. We open a fresh one here.
# silver_vault was persisted to disk by Phase 1g so it is available immediately.
# ==============================================================================
con = config.get_duckdb_conn()

try:
    # Verify silver_vault is accessible before proceeding
    vault_check = con.execute("SELECT COUNT(*) as n FROM silver_vault").pl()
    print(f"   ✅ silver_vault accessible: {vault_check['n'].item():,} records")

    
    # ==============================================================================
    # 3. FORENSIC LABEL FREQUENCY ANALYSIS
    # ==============================================================================
    # We query the Silver Vault to build a frequency profile of all ICD-10 labels.
    # This serves as a sanity check against the MedSynth paper's stated design:
    # uniform sampling of exactly 5 dialogue-note pairs per ICD-10 code.
    # Expected result: 0 rare labels (freq < 2), minimum frequency of 5.
    # Any deviation would indicate a data loading issue or an undocumented change
    # to the HuggingFace release.
    # ==============================================================================
    
    
    print("\n🕵️  Running Forensic Label Frequency Analysis...")

    label_stats = con.execute("""
        SELECT
            COUNT(DISTINCT label) as unique_codes,
            COUNT(*)              as total_records
        FROM silver_vault
    """).pl()

    rare_labels = con.execute("""
        SELECT label, COUNT(*) as freq
        FROM silver_vault
        GROUP BY label
        HAVING freq < 2
        ORDER BY freq ASC
    """).pl()

    print(f"   📊 Total unique labels:     {label_stats['unique_codes'].item():,}")
    print(f"   ⚠️  Rare labels (freq < 2): {len(rare_labels):,}")

    config.log_event(
        phase="Phase 2a: Biological Audit",
        action="label_frequency_analysis_complete",
        details={
            "total_unique_codes": label_stats['unique_codes'].item(),
            "rare_label_count": len(rare_labels),
        }
    )

    # ==============================================================================
    # 4. CANONICAL DECIMAL RESTORATION
    # ==============================================================================
    # Raw MedSynth codes omit the ICD-10 decimal point (e.g. M25562, N390).
    # We restore the canonical CDC format by inserting a decimal after the third
    # character (e.g. M25562 → M25.562, N390 → N39.0).
    #
    # IMPORTANT — 3-character codes are a special case:
    #   Codes like R32, J18, L80 are category-level parent codes (Noisy 111).
    #   They have no subclassification component, so inserting a decimal produces
    #   a trailing dot (e.g. R32.) which is invalid. These codes are left as-is,
    #   which matches their representation in the CDC FY2026 reference.
    #
    # This restoration is a canonicalisation step only — it does not alter which
    # codes are considered billable or non-billable (that was determined in Phase 1b).
    #
    # Placeholder-X codes (e.g. T781XXA, identified in Phase 1b) will receive a
    # decimal after position 3 (e.g. T78.1XXA). These are structurally unusual
    # but the transformation is applied consistently.
    # ==============================================================================
    print("\n🔬 Restoring ICD-10 Decimal Points...")

    # Step 1: Extract the first code from the label list.
    # Phase 1g confirmed this dataset is single-code per record, so list.first()
    # is safe and correct here.
    df_gold = df_valid.with_columns([
        pl.col("label").list.first().fill_null("").alias("raw_code")
    ])

    # Step 2: Conditionally insert decimal after 3rd character.
    #
    # Rule:
    #   len > 3  →  insert decimal  (e.g. M25562 → M25.562, N390 → N39.0)
    #   len == 3 →  leave as-is     (e.g. R32 → R32, J18 → J18)
    #   len == 0 →  leave as-is     (empty string — caught by validation below)
    #
    # NOTE: This must be a separate with_columns() call. Polars requires raw_code
    # to exist as a materialised column before it can be referenced in the next
    # expression — it cannot be defined and referenced in the same call.
    df_gold = df_gold.with_columns([
        pl.when(pl.col("raw_code").str.len_chars() > 3)
        .then(
            pl.col("raw_code").str.slice(0, 3) + "." + pl.col("raw_code").str.slice(3)
        )
        .otherwise(pl.col("raw_code"))
        .alias("standard_icd10")
    ])

    # Step 3: Validate restoration results.
    # A valid standard_icd10 must match one of two patterns:
    #   - Canonical format:   3 chars + decimal + 1+ chars  (e.g. M25.562, N39.0)
    #   - Parent code format: exactly 3 alphanumeric chars   (e.g. R32, J18)
    # Anything else (empty string, trailing dot, malformed) is flagged as invalid.
    validation_check = df_gold.with_columns(
        (
            pl.col("standard_icd10").str.contains(r"^\w{3}\.\w+$") |  # canonical
            pl.col("standard_icd10").str.contains(r"^\w{3}$")          # parent code
        ).alias("has_valid_format")
    )

    valid_format_pct = (
        validation_check.select(pl.col("has_valid_format").mean() * 100).item()
    )

    # Count each format type for the detailed audit summary
    canonical_count = validation_check.filter(
        pl.col("standard_icd10").str.contains(r"^\w{3}\.\w+$")
    ).height
    parent_count = validation_check.filter(
        pl.col("standard_icd10").str.contains(r"^\w{3}$")
    ).height
    invalid_count = validation_check.filter(
        ~pl.col("has_valid_format")
    ).height

    print(f"   ✅ Decimal restoration complete")
    print(f"   ✅ Valid format percentage:  {valid_format_pct:.1f}%")
    print(f"      ├─ Canonical (X##.###):  {canonical_count:,}")
    print(f"      ├─ Parent codes (X##):   {parent_count:,}")
    print(f"      └─ Invalid / empty:      {invalid_count:,}")

    # Surface any remaining invalids for inspection
    if invalid_count > 0:
        print(f"\n⚠️  INVALID FORMAT SAMPLE (first 10):")
        invalid_sample = validation_check.filter(
            ~pl.col("has_valid_format")
        ).select(["raw_code", "standard_icd10"]).unique().head(10)
        try:
            from IPython.display import display
            display(invalid_sample)
        except ImportError:
            print(invalid_sample)

    # ==============================================================================
    # 5. REGISTER GOLD LAYER
    # ==============================================================================
    print("\n💾 Registering Gold Layer...")

    config.register_dataframe("gold_medsynth", df_gold, phase="Phase 2a")
    print(f"   ✅ Gold Layer registered: {len(df_gold):,} records")

    # ==============================================================================
    # 6. AUDIT VISIBILITY: GOLD LAYER PREVIEW
    # ==============================================================================
    print("\n👁️  Gold Layer Preview (First 10 Records):")
    try:
        from IPython.display import display
        display(df_gold.select(["id", "label", "raw_code", "standard_icd10"]).head(10))
    except ImportError:
        print(df_gold.select(["id", "label", "raw_code", "standard_icd10"]).head(10))

    # ==============================================================================
    # 7. LOG THE TRANSFORMATION
    # ==============================================================================
    config.log_event(
        phase="Phase 2a: Biological Audit",
        action="decimal_restoration_complete",
        details={
            "total_records": len(df_gold),
            "valid_format_pct": round(valid_format_pct, 2),
            "canonical_format_count": canonical_count,
            "parent_code_count": parent_count,
            "invalid_format_count": invalid_count,
            "rare_label_count": len(rare_labels),
            "gold_artifact": "gold_medsynth"
        }
    )

    print(f"\n📝 Audit trail updated")

    # ==============================================================================
    # 8. ZERO-TRUST VERDICT
    # ==============================================================================
    print(f"\n🛡️  ZERO-TRUST VERDICT:")

    if valid_format_pct == 100.0:
        print(f"   ✅ 100% of codes have valid decimal format")
        print(f"   ✅ Gold layer is ready for downstream transformations")
    elif valid_format_pct >= 95:
        print(f"   ✅ {valid_format_pct:.1f}% of codes have valid decimal format")
        print(f"   ℹ️  {invalid_count} records with invalid format — review sample above")
        print(f"   ✅ Gold layer is ready for downstream transformations")
    elif valid_format_pct >= 80:
        print(f"   ⚠️  {valid_format_pct:.1f}% of codes have valid decimal format")
        print(f"   ⚠️  Review records with invalid formats before proceeding")
    else:
        print(f"   ❌ {valid_format_pct:.1f}% of codes have valid decimal format")
        print(f"   ❌ CRITICAL: Decimal restoration failed. Review Phase 1e validation.")

    # ==============================================================================
    # 9. AUDIT SUMMARY
    # ==============================================================================
    print(f"\n📊 PHASE 2a AUDIT SUMMARY")
    print(f"   " + "─" * 50)
    print(f"   📊 Total Records:              {len(df_gold):,}")
    print(f"   ✅ Valid Decimal Format:       {valid_format_pct:.1f}%")
    print(f"      ├─ Canonical (X##.###):    {canonical_count:,}")
    print(f"      └─ Parent codes (X##):     {parent_count:,}")
    print(f"   ⚠️  Invalid / empty:           {invalid_count:,}")
    print(f"   ⚠️  Rare Labels (freq < 2):    {len(rare_labels):,}")
    print(f"   🏅 Gold Artifact:              gold_medsynth")
    print(f"   " + "─" * 50)
    print(f"\n✅ PHASE 2a COMPLETE: Gold layer registered and validated")

finally:
    con.close()
    print("🦆 DuckDB connection for Phase 2a closed.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 2b: Gold Layer Discovery Auditor

Following decimal restoration in Phase 2a, we launch a browser-native auditor
to visually verify the Gold layer output on a record-by-record basis. This is a
human-in-the-loop confirmation step before any downstream modelling work begins.

### Audit Objectives

1. **Decimal Restoration HUD:** Each record displays the raw label (e.g. `M25562`)
   alongside the canonical Gold label (e.g. `M25.562`), allowing direct visual
   confirmation that the restoration logic applied correctly. 3-character parent
   codes (e.g. `R32`) are shown without a decimal, which is their correct canonical
   form.

2. **Billability Status:** Records are flagged based on their Phase 1b
   classification — billable leaf codes, Noisy 111 parent codes, or placeholder-X
   codes. This connects the visual audit to the validated CDC reference findings
   rather than relying on frequency thresholds.

3. **Column-Key Recovery:** The auditor implements case-insensitive column lookup
   to locate clinical text fields regardless of case-shifting during Polars-to-pandas
   conversion, preventing silent failures on column access.

4. **Ink-Black Clinical Focus:** High-contrast monospace rendering ensures SOAP
   section headers remain legible, allowing confirmation that diagnostic signal
   in the Assessment section is intact after Gold layer promotion.

> **Result:** A record-level visual audit of the Gold layer confirming that decimal
> restoration is correct and clinical content is preserved before proceeding to
> downstream transformation phases.

</div>

In [ ]:
# ==============================================================================
# PHASE 2b: GOLD LAYER DISCOVERY AUDITOR (INK-BLACK)
# ==============================================================================

import panel as pn
import numpy as np
import pandas as pd
from datetime import datetime

pn.extension(design='material')


def launch_biological_auditor(df_gold, rare_labels_pl):
    df_pd = df_gold.to_pandas()

    # --- COLUMN NAME RECOVERY LOGIC ---
    # Case-insensitive lookup guards against column case-shifting during
    # Polars → pandas conversion (e.g. 'Note' becoming 'note').
    col_map  = {c.lower(): c for c in df_pd.columns}
    note_key = col_map.get('note',      'Note')
    dial_key = col_map.get('dialogue',  'Dialogue')

    index_slider = pn.widgets.IntSlider(
        name='Examine Gold Layer Record (Index)',
        start=0, end=len(df_pd) - 1, value=51,
        bar_color='#8e44ad', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def get_audit_view(index):
        row = df_pd.iloc[index]

        raw_label      = row.get('label',        'N/A')
        standard_label = row.get('standard_icd10','N/A')
        note_content   = row.get(note_key,  "⚠️ Note column not found.")
        dial_content   = row.get(dial_key,  "⚠️ Dialogue column not found.")

        # Safely convert label to display string
        if isinstance(raw_label, (list, np.ndarray)):
            raw_label_str = ', '.join(str(c) for c in raw_label) if len(raw_label) > 0 else '[]'
        elif pd.isna(raw_label):
            raw_label_str = 'N/A'
        else:
            raw_label_str = str(raw_label)

        # Determine decimal restoration status for the HUD
        std = str(standard_label)
        import re
        if re.match(r'^\w{3}\.\w+$', std):
            restore_status  = "✅ CANONICAL FORMAT"
            restore_color   = "#27ae60"
        elif re.match(r'^\w{3}$', std):
            restore_status  = "ℹ️ PARENT CODE (NO DECIMAL)"
            restore_color   = "#2980b9"
        else:
            restore_status  = "⚠️ REVIEW REQUIRED"
            restore_color   = "#e74c3c"

        mapping_hud = f"""
        <div style="display: flex; gap: 20px; max-width: 850px; margin-bottom: 20px;">
            <div style="flex: 1; padding: 15px; background: #f8f9fa;
                        border: 2px solid #ddd; border-radius: 8px; text-align: center;">
                <span style="font-size: 0.8rem; color: #666; font-weight: bold;
                             text-transform: uppercase;">Raw Label</span><br>
                <span style="font-size: 1.8rem; color: #333;
                             font-weight: 900;">{raw_label_str}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: #fff;
                        border: 2px solid #2980b9; border-radius: 8px; text-align: center;">
                <span style="font-size: 0.8rem; color: #2980b9; font-weight: bold;
                             text-transform: uppercase;">Canonical (Gold)</span><br>
                <span style="font-size: 1.8rem; color: #2980b9;
                             font-weight: 900;">{standard_label}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: {restore_color};
                        border-radius: 8px; text-align: center; color: white;">
                <span style="font-size: 0.8rem; font-weight: bold;
                             text-transform: uppercase;">Restoration Status</span><br>
                <span style="font-size: 1.1rem; font-weight: 700;">{restore_status}</span>
            </div>
        </div>
        """

        ink_black_style = """
        <style>
            .audit-text-area {
                white-space: pre-wrap !important;
                word-wrap: break-word !important;
                font-family: 'Courier New', monospace !important;
                font-size: 1.0rem !important;
                font-weight: 700 !important;
                color: #000000 !important;
                background-color: #ffffff !important;
                padding: 20px;
                border: 1px solid #000;
                line-height: 1.5;
                -webkit-font-smoothing: none;
            }
        </style>
        """

        content = f'{ink_black_style}<div class="audit-text-area">{note_content}</div>'

        return pn.Column(
            pn.pane.HTML(mapping_hud, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 📝 Clinical Note (Gold Layer)",
                    pn.pane.HTML(content),
                    height=600, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Source Dialogue",
                    pn.pane.Str(
                        dial_content,
                        styles={'white-space': 'pre-wrap', 'font-weight': '500'}
                    ),
                    height=600, scroll=True, sizing_mode='stretch_width'
                )
            )
        )

    layout = pn.Column(
        "# 🔬 Gold Layer Discovery Auditor — Phase 2b\n"
        "**Objective:** Visually confirm decimal restoration and clinical content "
        "integrity across the Gold layer before downstream transformation.",
        index_slider,
        get_audit_view,
        sizing_mode='stretch_width'
    )

    return layout.show(title="MedSynth Gold Layer Auditor — Phase 2b", threaded=True)


# ==============================================================================
# LAUNCH AUDITOR
# Uses df_gold and rare_labels produced by Phase 2a
# ==============================================================================
if 'df_gold' not in locals():
    print("❌ df_gold not found. Please run Phase 2a first.")
else:
    # rare_labels comes from Phase 2a's DuckDB query
    if 'rare_labels' not in locals():
        print("⚠️  rare_labels not found — creating empty placeholder.")
        import polars as pl
        rare_labels_pl = pl.DataFrame({"label": [], "freq": []})
    else:
        rare_labels_pl = rare_labels

    print("🚀 Launching Gold Layer Discovery Auditor (Phase 2b)...")
    print("   💡 TIP: Use the slider to navigate records.")
    print("   💡 TIP: Verify canonical format (X##.###) for 4+ char codes.")
    print("   💡 TIP: Parent codes (3 chars, no decimal) are correct as-is.")

    bio_server = launch_biological_auditor(df_gold, rare_labels_pl)

    # ==============================================================================
    # AUDIT TRAIL
    # ==============================================================================
    config.log_event(
        phase="Phase 2b: Gold Layer Auditor",
        action="auditor_launched",
        details={
            "total_records": len(df_gold),
            "rare_labels_count": len(rare_labels_pl) if rare_labels_pl is not None else 0,
        }
    )

    print(f"\n📝 Audit trail updated")
    print(f"✅ Phase 2b COMPLETE: Gold layer auditor launched")

    print(f"\n🛡️  ZERO-TRUST VERDICT:")
    if len(rare_labels_pl) > 0:
        print(f"   ⚠️  {len(rare_labels_pl):,} rare labels identified — review before proceeding")
    else:
        print(f"   ✅ No rare labels found — all codes meet minimum frequency threshold")
        print(f"   ✅ Consistent with MedSynth uniform sampling design (5 per code)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif; border: 2px solid #e74c3c; padding: 20px; border-radius: 8px;">

## 🚨 Forensic Alert: Diagnostic Leakage Detected

During the **Phase 3a.1 Biological Audit**, visual inspection of the clinical narratives (Index 51) revealed a critical threat to model integrity: **Explicit Label Leakage** [cite: 2026-02-12, 2026-02-23].

### The Discovery
In the audited sample for **E66.9 (Obesity, unspecified)**, the raw clinical text explicitly contains the diagnostic answer within the `**3. Assessment:**` block [cite: 2026-02-23]:
> *"...Diagnosis: Obesity, unspecified (ICD-10: E66.9)"*

![Forensic Evidence of ICD-10 Leakage](./resources/images/icd-10_code_leakage.png)

### The Risk: "The Shortcut Trap"
If left unaddressed, ClinicalBERT will ignore the clinical evidence in the **Subjective** and **Objective** sections and simply "cheat" by extracting the code directly from the text [cite: 2026-02-12]. This leads to:
1. **Inflated Accuracy Metrics**: 99%+ accuracy in training that collapses in real-world deployment where codes aren't pre-written [cite: 2026-01-29].
2. **Zero Clinical Reasoning**: The model fails to learn the relationship between symptoms (e.g., BMI, joint pain) and the diagnosis [cite: 2026-02-12].

### Surgical Mitigation Required
This confirms the absolute necessity of the **Phase 5 Greedy Scrub** [cite: 2026-02-12]. We must programmatically redact:
* **Literal ICD-10 Codes**: Any string matching the `[A-Z][0-9][0-9]` pattern.
* **Semantic Labels**: The exact diagnostic string (e.g., "Obesity, unspecified") when it appears in the assessment block.

> **🛡️ Auditor Verdict**: The APSO-flip moves this leaked signal to Token 0, making it even easier for the model to "cheat." **Redaction is no longer optional; it is a prerequisite for a valid training run** [cite: 2026-01-27, 2026-02-23].

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🏷️ Phase 2c: Gold Layer Status Annotation

With the Gold layer persisted in Phase 2a and visually verified in Phase 2b, we
now formally embed the ICD-10 validation findings from Phase 1b directly into the
Gold layer as a first-class column. This closes the loop on the Noisy 111 analysis
and gives every downstream phase the information needed to make its own informed
decision about how to handle each code category.

No records are removed or modified. The annotation is additive only.

---

### The Three Status Categories

Every record in the Gold layer receives one of three status values, derived
directly from the Phase 1b CDC FY2026 reference join:

| Status | Description | Count | % |
|---|---|---|---|
| `billable` | Confirmed leaf code in CDC FY2026 tabular reference | 9,660 | 94.3% |
| `noisy_111` | Valid ICD-10 parent code, too broad for billing | 555 | 5.4% |
| `placeholder_x` | Structurally valid ICD-10-CM injury code, absent from CDC descriptions file | 25 | 0.24% |

The `noisy_111` and `placeholder_x` classifications are **not errors**. As
established in Phase 1b and confirmed by the MedSynth paper, the Noisy 111 codes
reflect real clinical coding practice from IQVIA insurance claims, and the
placeholder-X codes are legitimate injury codes not expanded in the CM tabular
reference.

---

### Why Annotate Rather Than Remove?

The appropriate treatment of non-billable codes depends on the downstream
modelling objective — which has not yet been fixed. Annotating preserves optionality:

* A **multi-class classification** task may want to exclude `noisy_111` records
  to avoid label ambiguity.
* A **hierarchical classification** task may want to retain them and use the
  parent-child ICD-10 structure explicitly.
* An **information extraction** task (e.g. extracting clinical entities) does not
  need billable codes at all and should retain everything.

By embedding `code_status` in the Gold layer now, any of these downstream
strategies can be implemented with a single filter — without rerunning the
CDC validation pipeline.

---

### Implementation

The status lookup is built from `df_checked` — the record-level validation
DataFrame produced by Phase 1b — and joined onto `df_gold` by `id`. The Phase 1b
three-bucket vocabulary is mapped to cleaner status labels:

- `"non_billable_parent (Noisy 111)"` → `"noisy_111"`
- `"invalid_or_malformed"` → `"placeholder_x"` *(more precise — these are not
  malformed, they are absent from the tabular reference)*

> **Result:** A fully annotated Gold layer where every record carries a
> validated, CDC-grounded `code_status` — forming the complete, audit-ready
> foundation for all downstream modelling phases.

</div>

In [ ]:
# ==============================================================================
# PHASE 2c: GOLD LAYER STATUS ANNOTATION
# ==============================================================================
# Purpose: Formally annotate every record in the Gold layer with its ICD-10
# validation status, derived from the Phase 1b CDC FY2026 reference join.
#
# This closes the loop on the Noisy 111 findings from Phase 1b by embedding
# the classification directly into the Gold layer as a first-class column,
# giving all downstream phases the information to make their own decisions
# about how to handle each category.
#
# Status values:
#   'billable'       — confirmed leaf code in CDC FY2026 tabular reference
#   'noisy_111'      — valid ICD-10 parent code, too broad for billing
#   'placeholder_x'  — structurally valid ICD-10-CM injury code, absent from
#                      CDC descriptions file (placeholder X convention)
# ==============================================================================

import polars as pl

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 2c Prerequisites...")

required = {'df_gold': df_gold, 'df_checked': df_checked, 'cdc_df': cdc_df}
for name, obj in required.items():
    if obj is None or (hasattr(obj, '__len__') and len(obj) == 0):
        raise ValueError(f"❌ CRITICAL: {name} is empty or None. Run Phase 1b and 2a first.")
    print(f"   ✅ {name} available: {len(obj):,} records")

# ==============================================================================
# 2. BUILD STATUS LOOKUP FROM PHASE 1b df_checked
# ==============================================================================
# df_checked (from Phase 1b) contains one row per code per record, with a
# 'status' column: 'billable', 'non_billable_parent (Noisy 111)',
# or 'invalid_or_malformed'.
#
# We take the first (and only) code per record — confirmed single-code dataset —
# and map Phase 1b's three-bucket classification to our cleaner status vocabulary.
# ==============================================================================
print("\n🔧 Building status lookup from Phase 1b validation results...")

STATUS_MAP = {
    "billable":                        "billable",
    "non_billable_parent (Noisy 111)": "noisy_111",
    "invalid_or_malformed":            "placeholder_x"
    # Note: 'invalid_or_malformed' in Phase 1b covers the 25 placeholder-X codes
    # confirmed as structurally valid ICD-10-CM codes absent from the CM tabular
    # reference. Renaming to 'placeholder_x' here is more precise and less
    # alarming — these are not errors.
}

# df_checked has columns: ID, code, code_no_decimal, description, is_parent, status
# We need one status per ID — take the first code row for each ID (single-code dataset)
status_lookup = (
    df_checked
    .with_columns(
        pl.col("status")
        .replace(list(STATUS_MAP.keys()), list(STATUS_MAP.values()))
        .alias("code_status")
    )
    .select(["ID", "code_status"])
    .unique(subset=["ID"], keep="first")
)

print(f"   ✅ Status lookup built: {len(status_lookup):,} records")

# Verify distribution matches Phase 1b findings
status_dist = status_lookup.group_by("code_status").agg(
    pl.len().alias("count")
).with_columns(
    (pl.col("count") / pl.col("count").sum() * 100).alias("pct")
).sort("count", descending=True)

print(f"\n📊 Status distribution (should match Phase 1b validation summary):")
try:
    from IPython.display import display
    display(status_dist)
except ImportError:
    print(status_dist)

# ==============================================================================
# 3. JOIN STATUS ONTO GOLD LAYER
# ==============================================================================
# df_gold has 'id' as String; status_lookup has 'ID' as Int64 (from df_raw).
# Cast to align types before joining.
# ==============================================================================
print("\n🔗 Joining status annotation onto Gold layer...")

df_gold_annotated = df_gold.join(
    status_lookup.with_columns(pl.col("ID").cast(pl.String).alias("id")),
    on="id",
    how="left"
)

# Verify no records lost and no nulls in code_status
join_check = df_gold_annotated.filter(pl.col("code_status").is_null()).height
if join_check > 0:
    print(f"   ⚠️  WARNING: {join_check:,} records have null code_status after join.")
    print(f"      This may indicate an ID mismatch between df_gold and df_checked.")
else:
    print(f"   ✅ Join complete: {len(df_gold_annotated):,} records, 0 null status values")

# ==============================================================================
# 4. UPDATE df_gold
# ==============================================================================

# Drop the redundant Int64 ID column introduced by the status join
df_gold = df_gold_annotated.drop("ID")

print(f"\n   ✅ df_gold updated with code_status column")
print(f"   📊 Gold layer schema: {df_gold.schema}")

# ==============================================================================
# 5. SUMMARY REPORT
# ==============================================================================
print(f"\n📊 PHASE 2c ANNOTATION SUMMARY")
print(f"   " + "─" * 50)

final_dist = df_gold.group_by("code_status").agg(
    pl.len().alias("count")
).with_columns(
    (pl.col("count") / pl.col("count").sum() * 100).alias("pct")
).sort("count", descending=True)

for row in final_dist.iter_rows(named=True):
    status  = row["code_status"]
    count   = row["count"]
    pct     = row["pct"]
    icon    = "✅" if status == "billable" else ("ℹ️ " if status == "noisy_111" else "🔍")
    print(f"   {icon} {status:20s}: {count:>6,} records ({pct:.1f}%)")

print(f"   " + "─" * 50)
print(f"   📊 Total:                  {len(df_gold):,} records")

# ==============================================================================
# 6. RE-REGISTER UPDATED GOLD LAYER
# ==============================================================================
print(f"\n💾 Re-registering annotated Gold layer...")
config.register_dataframe("gold_medsynth", df_gold, phase="Phase 2c")
print(f"   ✅ gold_medsynth updated with code_status annotation")

# ==============================================================================
# 7. AUDIT TRAIL
# ==============================================================================
billable_count    = df_gold.filter(pl.col("code_status") == "billable").height
noisy_count       = df_gold.filter(pl.col("code_status") == "noisy_111").height
placeholder_count = df_gold.filter(pl.col("code_status") == "placeholder_x").height

config.log_event(
    phase="Phase 2c: Gold Layer Status Annotation",
    action="status_annotation_complete",
    details={
        "total_records":       len(df_gold),
        "billable_count":      billable_count,
        "billable_pct":        round(billable_count / len(df_gold) * 100, 2),
        "noisy_111_count":     noisy_count,
        "noisy_111_pct":       round(noisy_count / len(df_gold) * 100, 2),
        "placeholder_x_count": placeholder_count,
        "placeholder_x_pct":   round(placeholder_count / len(df_gold) * 100, 2),
        "annotation_source":   "Phase 1b CDC FY2026 validation (df_checked)",
        "null_status_records": join_check
    }
)

print(f"\n📝 Audit trail updated")

# ==============================================================================
# 8. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")
print(f"   ✅ All {len(df_gold):,} Gold layer records carry a validated status")
print(f"   ✅ Status derived from CDC FY2026 reference join (Phase 1b)")
print(f"   ℹ️  Downstream phases should filter or weight by code_status")
print(f"      as appropriate for their specific modelling objectives.")
print(f"\n✅ PHASE 2c COMPLETE: Gold layer fully annotated and re-registered")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## ✅ Pipeline Complete: The Annotated Gold Layer

This concludes the forensic ingestion and validation pipeline. Rather than
pruning records based on a rejected semantic recovery experiment, we have taken
a more rigorous approach — annotating every record with its validated ICD-10
status and preserving the full dataset for downstream use.

### What the Pipeline Produced

Starting from 10,240 raw MedSynth records, the pipeline has delivered a fully
validated, annotated Gold layer with the following properties:

| Property | Value |
|---|---|
| Total records | 10,240 |
| Billable (CDC FY2026 leaf codes) | 9,660 (94.3%) |
| Noisy 111 (valid parent codes) | 555 (5.4%) |
| Placeholder-X (injury codes) | 25 (0.24%) |
| SOAP extraction success | 100% |
| Canonical ICD-10 format | 100% |
| Empty dialogue records | 2 (sentinel imputed) |

### On the Noisy 111 Records

The 555 records carrying non-billable parent codes are **not errors**. The
MedSynth authors selected codes from the top 2,001 most frequent ICD-10 codes
in real IQVIA insurance claims data, and clinicians genuinely submit
category-level codes in practice. These records are retained in the Gold layer
with a `code_status` of `noisy_111` — downstream phases can include or exclude
them based on their specific modelling requirements.

### What Each Record Carries

Every record in `df_gold` / `gold_medsynth` now contains:

* Original clinical note and doctor-patient dialogue
* Raw ICD-10 label array (decimal-free, as stored in source data)
* Canonical `standard_icd10` string (decimal-restored, CDC FY2026 format)
* Extracted SOAP sections (Subjective, Objective, Assessment, Plan)
* `code_status` classification grounded in CDC FY2026 reference join

### Next Steps

Downstream phases should select records appropriate to their task:
```python
# Billable records only (strictest label quality)
df_billable = df_gold.filter(pl.col("code_status") == "billable")  # 9,660 records

# All records including parent codes (maximum data)
df_all = df_gold  # 10,240 records

# Exclude only the placeholder-X codes
df_no_placeholders = df_gold.filter(pl.col("code_status") != "placeholder_x")  # 10,215 records
```

> **Result:** A transparent, audit-ready Gold layer where every design decision
> is documented, every record is traceable to its source, and no data has been
> irreversibly discarded.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔄 Phase 3a: APSO-Flip (Diagnostic Signal Prioritisation)

With the Gold layer fully annotated in Phase 2c, we now address the truncation
problem quantified in Phase 1c: 64.7% of clinical notes exceed ClinicalBERT's
512-token context window, and in standard SOAP format the Assessment section —
which contains the primary diagnostic signal — appears last and is routinely
truncated away.

### The APSO-Flip

The transformation reorders the four SOAP sections into **APSO format**:
**Assessment → Plan → Subjective → Objective**. This places the diagnostic
signal at Token 0, guaranteeing its survival through the model's context window
regardless of note length.

The Gold layer already contains pre-extracted SOAP sections as individual
columns (`assessment`, `plan`, `subjective`, `objective`), produced by the
Phase 1e Pydantic Gatekeeper with 100% extraction success. The APSO-Flip
recomposes these directly — no regex re-processing of the raw note is required.
This is more robust and avoids re-introducing section boundary detection
problems the Gatekeeper already solved.

### What Changes, What Doesn't

* **Added:** `apso_note` column — Assessment-first recomposition of the note
* **Added:** `apso_token_estimate` — estimated token count for the APSO note
* **Preserved:** `note` column — original SOAP note retained for full audit
  traceability and for tasks where the original ordering is preferred
* **Unchanged:** All other columns including `code_status`, `standard_icd10`,
  `label`, and all extracted SOAP section columns

The APSO-Flip does not reduce token count — the same content is present, just
reordered. The transformation protects the Assessment from truncation by moving
it to the front; it does not compress the note.

### Scope

The transformation is applied to all 10,240 Gold layer records regardless of
`code_status`. Downstream phases should select records by `code_status` as
appropriate for their modelling objective:
```python

In [ ]:
# ==============================================================================
# PHASE 3a: APSO-FLIP (DIAGNOSTIC SIGNAL PRIORITISATION)
# ==============================================================================
# Purpose: Reorder each clinical note from standard SOAP format into APSO format
# (Assessment, Plan, Subjective, Objective), ensuring the primary diagnostic
# signal appears at Token 0 and survives ClinicalBERT's 512-token context window.
#
# Approach: The Gold layer already contains pre-extracted SOAP sections as
# individual columns (assessment, plan, subjective, objective), produced by the
# Phase 1e Pydantic Gatekeeper. We recompose these directly rather than
# re-running regex on the raw note — this is more robust, faster, and avoids
# re-introducing the section boundary detection problem the Gatekeeper already
# solved for all 10,240 records.
#
# Dataset scope: The full Gold layer (10,240 records) is transformed.
# Downstream phases can filter by code_status as appropriate for their task.
# ==============================================================================

import polars as pl

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 3a Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    print("❌ CRITICAL: df_gold not found. Please run Phase 2c first.")
    raise ValueError("df_gold not available in local scope")

required_cols = {'assessment', 'plan', 'subjective', 'objective', 'note', 'code_status'}
missing_cols  = required_cols - set(df_gold.columns)
if missing_cols:
    raise ValueError(
        f"❌ CRITICAL: df_gold is missing required columns: {missing_cols}\n"
        f"   Please run Phase 1e (Pydantic Firewall) and Phase 2c (Status Annotation) first."
    )

print(f"   ✅ df_gold available: {len(df_gold):,} records")
print(f"   ✅ All required SOAP columns present")

# Confirm SOAP extraction coverage before transforming
soap_coverage = {
    section: df_gold.filter(pl.col(section).is_not_null()).height
    for section in ['subjective', 'objective', 'assessment', 'plan']
}
print(f"\n📊 SOAP Column Coverage (pre-transformation):")
for section, count in soap_coverage.items():
    pct = count / len(df_gold) * 100
    print(f"   {section.capitalize():12s}: {count:,}/{len(df_gold):,} ({pct:.1f}%)")

# ==============================================================================
# 2. APSO RECOMPOSITION
# ==============================================================================
# Reorder sections: Assessment → Plan → Subjective → Objective
#
# Rationale (from Phase 1c findings):
#   - 64.7% of Notes exceed 512 tokens
#   - Standard SOAP places Assessment at the end — it is routinely truncated
#   - Placing Assessment at Token 0 guarantees diagnostic signal survival
#     regardless of note length
#
# Null handling: fill_null("") ensures records with missing sections still
# produce a valid apso_note rather than propagating nulls.
#
# Note: This transformation does not modify the original 'note' column.
# Both the original SOAP note and the recomposed APSO note are retained
# in the Gold layer for full audit traceability.
# ==============================================================================
print("\n🔄 Applying APSO-Flip transformation...")

df_gold = df_gold.with_columns([
    (
        pl.col("assessment").fill_null("") + "\n\n" +
        pl.col("plan").fill_null("")       + "\n\n" +
        pl.col("subjective").fill_null("") + "\n\n" +
        pl.col("objective").fill_null("")
    ).str.strip_chars().alias("apso_note")
])

# Verify no null apso_note values
null_apso = df_gold.filter(pl.col("apso_note").is_null()).height
empty_apso = df_gold.filter(pl.col("apso_note") == "").height

if null_apso > 0 or empty_apso > 0:
    print(f"   ⚠️  WARNING: {null_apso} null and {empty_apso} empty apso_note values.")
    print(f"      Review SOAP extraction coverage in Phase 1e.")
else:
    print(f"   ✅ APSO recomposition complete: 0 null, 0 empty values")

# ==============================================================================
# 3. TOKEN PRESSURE AUDIT (POST-FLIP)
# ==============================================================================
# Estimate token counts for the recomposed APSO notes using the same
# words × 1.3 heuristic established in Phase 1c.
#
# The APSO-Flip does not reduce token count — the same content is present,
# just reordered. The purpose is not compression but prioritisation: ensuring
# the Assessment section occupies the first tokens in the model's context window.
# ==============================================================================
print("\n📊 Token Pressure Audit (APSO Notes)...")

TOKEN_HEURISTIC = 1.3
BERT_LIMIT      = 512

df_gold = df_gold.with_columns([
    (
        pl.col("apso_note")
        .str.count_matches(r"\S+")
        .fill_null(0)
        * TOKEN_HEURISTIC
    ).cast(pl.Float64).alias("apso_token_estimate")
])

total_records    = len(df_gold)
at_risk          = df_gold.filter(pl.col("apso_token_estimate") > BERT_LIMIT).height
within_window    = total_records - at_risk
at_risk_pct      = at_risk / total_records * 100
within_window_pct = within_window / total_records * 100

avg_tokens = df_gold.select(pl.col("apso_token_estimate").mean()).item()
max_tokens = df_gold.select(pl.col("apso_token_estimate").max()).item()

print(f"   📈 Average estimated tokens:  {avg_tokens:.0f}")
print(f"   📈 Maximum estimated tokens:  {max_tokens:.0f}")
print(f"   ✅ Within 512-token window:   {within_window:,} ({within_window_pct:.1f}%)")
print(f"   ⚠️  Exceeding 512-token limit: {at_risk:,} ({at_risk_pct:.1f}%)")
print(f"\n   ℹ️  Note: Token count is unchanged by the APSO-Flip.")
print(f"      The flip protects the Assessment section from truncation")
print(f"      by moving it to Token 0 — it does not reduce total length.")

# ==============================================================================
# 4. ASSESSMENT SECTION PREVIEW
# ==============================================================================
# Confirm Assessment content appears at the start of apso_note for the
# first few records — visual spot-check of the recomposition.
# ==============================================================================
print("\n👁️  APSO Note Preview (First 3 Records — Assessment Section Start):")
preview = df_gold.select(["id", "standard_icd10", "apso_note"]).head(3)
for row in preview.iter_rows(named=True):
    apso_preview = row["apso_note"][:200].replace("\n", " ↵ ")
    print(f"\n   ID: {row['id']} | Code: {row['standard_icd10']}")
    print(f"   {apso_preview}...")

# ==============================================================================
# 5. CODE STATUS BREAKDOWN OF TRANSFORMED RECORDS
# ==============================================================================
print(f"\n📊 Code Status Breakdown (post-APSO transformation):")
status_breakdown = df_gold.group_by("code_status").agg(
    pl.len().alias("count")
).with_columns(
    (pl.col("count") / pl.col("count").sum() * 100).alias("pct")
).sort("count", descending=True)

for row in status_breakdown.iter_rows(named=True):
    icon = "✅" if row["code_status"] == "billable" else (
           "ℹ️ " if row["code_status"] == "noisy_111" else "🔍")
    print(f"   {icon} {row['code_status']:20s}: {row['count']:>6,} ({row['pct']:.1f}%)")

# ==============================================================================
# 6. REGISTER UPDATED GOLD LAYER
# ==============================================================================
print(f"\n💾 Registering APSO-flipped Gold layer...")
config.register_dataframe("gold_medsynth", df_gold, phase="Phase 3a")
print(f"   ✅ gold_medsynth updated: {len(df_gold):,} records, "
      f"{len(df_gold.columns)} columns")

# ==============================================================================
# 7. AUDIT TRAIL
# ==============================================================================
config.log_event(
    phase="Phase 3a: APSO-Flip",
    action="apso_transformation_complete",
    details={
        "total_records":          len(df_gold),
        "null_apso_notes":        null_apso,
        "empty_apso_notes":       empty_apso,
        "avg_apso_tokens":        round(avg_tokens, 1),
        "max_apso_tokens":        round(max_tokens, 1),
        "records_within_window":  within_window,
        "records_exceeding_limit": at_risk,
        "records_exceeding_pct":  round(at_risk_pct, 2),
        "token_heuristic":        TOKEN_HEURISTIC,
        "bert_limit":             BERT_LIMIT,
        "transformation_method":  "pre-extracted SOAP columns (Phase 1e Gatekeeper)",
        "original_note_preserved": True
    }
)

print(f"\n📝 Audit trail updated")

# ==============================================================================
# 8. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")
print(f"   ✅ APSO-Flip applied to all {len(df_gold):,} records")
print(f"   ✅ Assessment section now at Token 0 for every record")
print(f"   ✅ Original SOAP note preserved in 'note' column")
print(f"   ℹ️  {at_risk:,} records ({at_risk_pct:.1f}%) still exceed 512 tokens —")
print(f"      their tail content will be truncated, but diagnostic signal")
print(f"      (Assessment) is now guaranteed to survive.")
print(f"\n✅ PHASE 3a COMPLETE: Gold layer APSO-flipped and re-registered")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

### 📊 Phase 3a Observations

**Token pressure is confirmed as the structural default for this dataset.** The
APSO-Flip token audit is consistent with both Phase 1c's empirical measurement
and the MedSynth paper's reported averages (932 tokens for dialogues, 621 for
notes) — 64.8% of APSO notes exceed the 512-token limit, which is expected given
the clinical depth of the generated narratives.

Three findings from the transformation are worth carrying forward:

1. **Assessment survival is now guaranteed.** With Assessment at Token 0, the
   primary diagnostic signal survives truncation for 100% of records regardless
   of note length. The 6,635 records that exceed the window will lose their
   Subjective and Objective tail content — clinically less critical for a
   classification task — but will retain the coded diagnosis.

2. **Markdown formatting is present in the APSO notes.** The preview shows `**`
   bold markers appearing at section boundaries. These are artefacts of the
   MedSynth Note Writer Agent's markdown output style and will be seen as tokens
   by ClinicalBERT's WordPiece tokeniser. A downstream preprocessing step should
   strip markdown formatting before tokenisation to avoid consuming context window
   space with non-clinical tokens.

3. **ICD-10 codes appear inline in the Assessment text.** The preview shows
   strings like "Pain in the left knee (ICD-10 code M25.562)" directly in the
   Assessment section. This is a known property of the MedSynth generation
   pipeline — the Note Writer Agent embedded the ICD-10 code in the narrative.
   Downstream models trained on this data will have access to the code string
   as a feature, which may inflate classification metrics. This should be
   accounted for in evaluation design.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔍 Phase 3b: ICD-10 Leakage Detection

The Phase 3a APSO note previews confirmed that the MedSynth Note Writer Agent
embedded ICD-10 code strings directly in the clinical narratives, typically
within the Assessment section (e.g., *"Diagnosis: Pain in left knee
(ICD-10: M25.562)"*).

This constitutes data leakage. A model trained on notes containing explicit code
strings can learn to predict labels by pattern-matching the code rather than
understanding the clinical narrative. This would inflate evaluation metrics and
produce a model that fails to generalise to real clinical notes, which never
contain ICD-10 codes in the text.

This phase quantifies the leakage precisely before any redaction is applied.
No records are modified here.

### Detection Objectives

1. **Explicit Code Leakage:** Scan the `apso_note` column for ICD-10 code
   strings using a pattern anchored on the ICD-10 structural convention — a
   leading letter followed by exactly two digits — which correctly discriminates
   ICD-10 codes from common medical abbreviations (MCL, CBC, TSH, HPI) that
   share a superficially similar alphanumeric structure. The pattern covers raw
   format (`M25562`), canonical format (`M25.562`), and placeholder-X codes
   (`T78.1XXA`). All 13 pattern validation tests passed before scanning.

2. **Assessment Column Leakage:** The Assessment section is the primary leakage
   source — the Note Writer Agent placed the diagnosis (including the code) in
   the Assessment section by design. Scanning this column separately establishes
   the scope of the primary leakage vector.

3. **Secondary Section Leakage:** Scan the Subjective, Objective, and Plan
   columns to determine whether leakage extends beyond the Assessment section,
   informing the targeted redaction strategy in Phase 3c.

### Why This Matters

The MedSynth paper notes that the dataset is designed for Dial-2-Note and
Note-2-Dial tasks, where the ICD-10 code is used as a generation conditioning
signal, not as a training label. Using this dataset for ICD-10 classification
without removing the embedded codes would create a trivial pattern-matching
task rather than a genuine clinical reasoning challenge.

### Observed Results

| Section | Leaking records | % of dataset |
|---|---|---|
| Assessment | 2,855 | 27.9% |
| Plan | 238 | 2.3% |
| Objective | 89 | 0.9% |
| Subjective | 60 | 0.6% |
| **Total unique records** | **2,923** | **28.5%** |

The Assessment section accounts for 97.7% of all leaking records, confirming
it as the primary and overwhelmingly dominant leakage vector. 71.5% of records
(7,317) contain no ICD-10 code strings and require no redaction.

> **Result:** A forensic leakage report confirming that 28.5% of records contain
> explicit ICD-10 code strings, concentrated almost entirely in the Assessment
> section. The redaction strategy in Phase 3c is targeted accordingly.

</div>

In [ ]:
# ==============================================================================
# PHASE 3b: LEAKAGE DETECTION (IDENTIFICATION ONLY)
# ==============================================================================
# Purpose: Quantify the extent of ICD-10 code leakage in the APSO notes before
# any redaction is applied. This phase adopts a "validate and confirm" mindset —
# we measure the problem precisely before attempting to fix it.
#
# Background: The MedSynth Note Writer Agent embedded ICD-10 codes directly
# in the clinical narratives (e.g., "Diagnosis: Pain in left knee (ICD-10:
# M25.562)"). A model trained on notes containing explicit code strings can
# learn to predict labels by pattern-matching the code rather than understanding
# the clinical narrative, inflating evaluation metrics and producing a model
# that fails to generalise to real clinical notes.
#
# PATTERN DESIGN NOTE:
# ICD-10 codes always begin with one letter followed by EXACTLY TWO DIGITS.
# This distinguishes them from common medical abbreviations (MCL, ACL, CBC,
# TSH, CMP, LFT) which begin with a letter followed by other letters — not digits.
# Using letter + two-digit anchor eliminates the false positives that arise
# from a purely alphanumeric pattern.
#
# Pattern covers:
#   Raw format (no decimal):  M25562, N390, T781XXA, J189
#   Canonical format:         M25.562, N39.0, T78.1XXA, J18.9
#   3-char parent codes:      J18, R32, N10
#
# This phase is identification only. No records are modified here.
# Redaction is applied in Phase 3c.
# ==============================================================================

import polars as pl
import re

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 3b Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    print("❌ CRITICAL: df_gold not found. Please run Phase 3a first.")
    raise ValueError("df_gold not available in local scope")

if 'apso_note' not in df_gold.columns:
    print("❌ CRITICAL: apso_note column not found. Please run Phase 3a (APSO-Flip) first.")
    raise ValueError("apso_note column not present in df_gold")

print(f"   ✅ df_gold available: {len(df_gold):,} records")
print(f"   ✅ apso_note column present")

# ==============================================================================
# 2. DEFINE LEAKAGE DETECTION PATTERN
# ==============================================================================
# Anchored on letter + TWO DIGITS to distinguish ICD-10 codes from medical
# abbreviations. The two-digit anchor is the critical discriminator:
#
#   ✅ Matches:  M25.562  M25562  N39.0  N390  J18  T78.1XXA  T781XXA
#   ❌ Rejects:  MCL  ACL  CBC  TSH  CMP  LFT  HPI  ROS  BMI
#
# Two branches:
#   Branch 1: Canonical (with decimal) — letter + 2 digits + decimal + 1-4 alphanum
#   Branch 2: Raw (no decimal)         — letter + 2 digits + 0-5 additional alphanum
#             (covers 3-char codes like J18 and longer codes like M25562)
# ==============================================================================
ICD10_LEAK_PATTERN = (
    r'\b[A-Z][0-9]{2}\.[0-9A-Z]{1,4}\b'   # canonical: M25.562, N39.0, T78.1XXA
    r'|'
    r'\b[A-Z][0-9]{2}[0-9A-Z]{0,5}\b'     # raw: M25562, N390, J18, T781XXA
)

print(f"\n🔍 Leakage detection pattern:")
print(f"   Canonical (with decimal): [A-Z][0-9]{{2}}\\.[0-9A-Z]{{1,4}}")
print(f"   Raw (no decimal):         [A-Z][0-9]{{2}}[0-9A-Z]{{0,5}}")
print(f"   ✅ Catches: M25.562, M25562, N39.0, N390, J18, T78.1XXA")
print(f"   ❌ Rejects: MCL, ACL, CBC, TSH, CMP, LFT, HPI, BMI")

# ==============================================================================
# 3. VALIDATE PATTERN ON KNOWN EXAMPLES
# ==============================================================================
# Quick sanity check before running across the full dataset
print(f"\n🧪 Pattern validation on known examples:")

test_cases = [
    ("M25.562",  True,  "canonical code"),
    ("M25562",   True,  "raw code"),
    ("N39.0",    True,  "canonical code"),
    ("N390",     True,  "raw code"),
    ("J18",      True,  "3-char parent code"),
    ("T78.1XXA", True,  "placeholder-X canonical"),
    ("T781XXA",  True,  "placeholder-X raw"),
    ("MCL",      False, "medical abbreviation"),
    ("ACL",      False, "medical abbreviation"),
    ("CBC",      False, "medical abbreviation"),
    ("TSH",      False, "medical abbreviation"),
    ("BMI",      False, "medical abbreviation"),
    ("HPI",      False, "medical abbreviation"),
]

all_pass = True
for term, expected, label in test_cases:
    matched = bool(re.search(ICD10_LEAK_PATTERN, term))
    status  = "✅" if matched == expected else "❌"
    if matched != expected:
        all_pass = False
    print(f"   {status} '{term:12s}' ({label:28s}) — {'matched' if matched else 'rejected'}")

if all_pass:
    print(f"\n   ✅ All pattern validation tests passed")
else:
    print(f"\n   ⚠️  Some pattern validation tests failed — review pattern before proceeding")

# ==============================================================================
# 4. DETECT EXPLICIT CODE LEAKAGE IN APSO NOTES
# ==============================================================================
print("\n🕵️  Scanning APSO notes for explicit ICD-10 code strings...")

df_explicit_leaks = df_gold.filter(
    pl.col("apso_note").str.contains(ICD10_LEAK_PATTERN)
)

explicit_leak_count = df_explicit_leaks.height
explicit_leak_pct   = explicit_leak_count / len(df_gold) * 100

print(f"   📊 Records with explicit ICD-10 codes in apso_note: "
      f"{explicit_leak_count:,} ({explicit_leak_pct:.1f}%)")

# ==============================================================================
# 5. DETECT LEAKAGE BY SECTION
# ==============================================================================
# Scan each SOAP section independently to understand the distribution of
# leakage and inform the targeted redaction strategy in Phase 3c.
# ==============================================================================
print("\n🕵️  Scanning individual SOAP sections...")

section_leakage = {}
for section in ['assessment', 'subjective', 'objective', 'plan']:
    leaks = df_gold.filter(
        pl.col(section).is_not_null() &
        pl.col(section).str.contains(ICD10_LEAK_PATTERN)
    ).height
    section_leakage[section] = leaks
    pct = leaks / len(df_gold) * 100
    print(f"   📊 {section.capitalize():12s}: {leaks:,} records ({pct:.1f}%)")

# ==============================================================================
# 6. SAMPLE LEAKING RECORDS FOR INSPECTION
# ==============================================================================
print(f"\n👁️  SAMPLE LEAKING RECORDS (First 5 — Assessment column):")

df_assessment_leaks = df_gold.filter(
    pl.col("assessment").is_not_null() &
    pl.col("assessment").str.contains(ICD10_LEAK_PATTERN)
)

if df_assessment_leaks.height > 0:
    sample_leaks = df_assessment_leaks.select([
        "id", "standard_icd10", "assessment"
    ]).head(5)

    for row in sample_leaks.iter_rows(named=True):
        assessment_preview = (row["assessment"] or "")[:200].replace("\n", " ↵ ")
        print(f"\n   ID: {row['id']} | Code: {row['standard_icd10']}")
        print(f"   Assessment: {assessment_preview}...")

# ==============================================================================
# 7. LEAKAGE SUMMARY REPORT
# ==============================================================================
assessment_leak_count = section_leakage['assessment']
assessment_leak_pct   = assessment_leak_count / len(df_gold) * 100

print(f"\n📊 PHASE 3b LEAKAGE DETECTION SUMMARY")
print(f"   " + "─" * 50)
print(f"   📊 Total records scanned:         {len(df_gold):,}")
print(f"   ⚠️  Explicit leaks (apso_note):   {explicit_leak_count:,} ({explicit_leak_pct:.1f}%)")
print(f"   " + "─" * 30)
print(f"   📊 Section breakdown:")
for section, count in section_leakage.items():
    pct  = count / len(df_gold) * 100
    icon = "⚠️ " if count > 0 else "✅"
    print(f"   {icon} {section.capitalize():12s}: {count:>6,} ({pct:.1f}%)")
print(f"   " + "─" * 50)
print(f"   ℹ️  No records modified in this phase.")
print(f"   ℹ️  Redaction will be applied in Phase 3c.")

# ==============================================================================
# 8. AUDIT TRAIL
# ==============================================================================
config.log_event(
    phase="Phase 3b: Leakage Detection",
    action="leakage_detection_complete",
    details={
        "total_records":           len(df_gold),
        "explicit_leak_count":     explicit_leak_count,
        "explicit_leak_pct":       round(explicit_leak_pct, 2),
        "assessment_leak_count":   section_leakage['assessment'],
        "subjective_leak_count":   section_leakage['subjective'],
        "objective_leak_count":    section_leakage['objective'],
        "plan_leak_count":         section_leakage['plan'],
        "detection_pattern":       ICD10_LEAK_PATTERN,
        "pattern_validation":      "passed" if all_pass else "failed",
        "records_modified":        False
    }
)

print(f"\n📝 Audit trail updated")
print(f"✅ PHASE 3b COMPLETE: Leakage quantified — redaction pending (Phase 3c)")

# ==============================================================================
# 9. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")

if explicit_leak_count == 0:
    print(f"   ✅ No explicit ICD-10 leakage detected")
    print(f"   ✅ Dataset may proceed to training without redaction")
else:
    print(f"   ⚠️  {explicit_leak_count:,} records ({explicit_leak_pct:.1f}%) contain "
          f"explicit ICD-10 code strings")
    print(f"   ⚠️  Training on these notes would allow the model to predict labels")
    print(f"      by code pattern-matching rather than clinical reasoning")
    print(f"   ✅ Redaction strategy to be applied in Phase 3c")
    dominant = max(section_leakage, key=section_leakage.get)
    print(f"   ℹ️  Primary leakage source: {dominant.capitalize()} column "
          f"({section_leakage[dominant]:,} records, "
          f"{section_leakage[dominant]/len(df_gold)*100:.1f}%)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🕵️ Phase 3b.1: Leakage Forensic Auditor

Before applying redaction in Phase 3c, we deploy an interactive browser-native
auditor to manually verify that the 2,923 records flagged by Phase 3b genuinely
contain explicit ICD-10 code strings — confirming that the detection pattern is
identifying real leakage rather than false positives.

This is a human-in-the-loop validation gate. No records are modified here.

### What to Look For

Navigate the slider across a representative sample of records and confirm that
the APSO note (left panel) visibly contains ICD-10 code strings in one of the
formats the Phase 3b pattern was designed to catch:

- `(ICD-10: M25.562)` — parenthetical with colon
- `(ICD-10 code N39.0)` — parenthetical with label
- `(M25.562)` — bare parenthetical
- `M25.562` — inline without parentheses

Sample across different `code_status` values — `billable`, `noisy_111`, and
`placeholder_x` — to confirm the leakage pattern is consistent across all
three categories. The `code_status` field is shown in the HUD for each record.

The source dialogue (right panel) provides the original doctor-patient
transcript for cross-reference, confirming that the clinical content is rich
and the leakage is confined to the administrative code string rather than
being a structural property of the narrative itself.

> **Validation Gate:** Once satisfied that the detected leaks are genuine,
> proceed to Phase 3c to apply redaction. The auditor should be stopped using
> the browser tab before continuing.

</div>

In [ ]:
# ==============================================================================
# PHASE 3b.1: LEAKAGE FORENSIC AUDITOR (VISUAL INSPECTION)
# ==============================================================================
# Purpose: Interactive browser-native audit of records identified as leaking
# in Phase 3b. Allows manual confirmation that detected leaks are genuine
# ICD-10 code strings before redaction is applied in Phase 3c.
#
# This is a human-in-the-loop validation gate between detection and redaction.
# It operates on df_explicit_leaks produced by Phase 3b.
# ==============================================================================

import panel as pn
pn.extension(design='material')


def launch_leakage_auditor(df_leaking_records):
    """
    Launches a browser-native auditor showing only the records identified
    as leaking in Phase 3b, for manual verification before redaction.
    """
    df_pd = df_leaking_records.to_pandas()

    if len(df_pd) == 0:
        print("✅ No leaks detected — auditor not required.")
        return None

    print(f"🔎 Launching auditor for {len(df_pd):,} leaking records...")

    index_slider = pn.widgets.IntSlider(
        name='Inspect Leaky Record (Index)',
        start=0, end=len(df_pd) - 1, value=0,
        bar_color='#f39c12', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def get_leakage_view(index):
        row        = df_pd.iloc[index]
        std_code   = row.get('standard_icd10', 'N/A')
        record_id  = row.get('id',             'N/A')
        code_status = row.get('code_status',   'N/A')
        apso_note  = row.get('apso_note',      row.get('note', 'N/A'))
        dialogue   = row.get('dialogue',       'N/A')

        mapping_hud = f"""
        <div style="display: flex; gap: 15px; max-width: 850px;
                    margin-bottom: 20px; font-family: sans-serif;">
            <div style="flex: 1; padding: 12px; background: #fff;
                        border: 3px solid #f39c12; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.75rem; color: #666; font-weight: bold;
                             text-transform: uppercase;">Record ID</span><br>
                <span style="font-size: 1.6rem; color: #333;
                             font-weight: 900;">{record_id}</span>
            </div>
            <div style="flex: 1; padding: 12px; background: #fff;
                        border: 3px solid #2980b9; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.75rem; color: #2980b9; font-weight: bold;
                             text-transform: uppercase;">ICD-10 Label</span><br>
                <span style="font-size: 1.6rem; color: #2980b9;
                             font-weight: 900;">{std_code}</span>
            </div>
            <div style="flex: 1; padding: 12px; background: #f39c12;
                        border-radius: 8px; text-align: center; color: white;">
                <span style="font-size: 0.75rem; font-weight: bold;
                             text-transform: uppercase;">Status</span><br>
                <span style="font-size: 1.1rem; font-weight: 800;">
                    ⚠️ LEAKAGE DETECTED</span>
                <div style="font-size: 0.8rem; margin-top: 5px;">
                    {code_status} &nbsp;|&nbsp;
                    Record {index + 1} of {len(df_pd):,}
                </div>
            </div>
        </div>
        """

        ink_style = """
        <style>
            .forensic-text {
                white-space: pre-wrap;
                font-family: 'Courier New', monospace;
                font-weight: 700;
                font-size: 0.95rem;
                padding: 20px;
                border: 2px solid #333;
                background-color: #fdfdfd;
                line-height: 1.5;
            }
        </style>
        """

        return pn.Column(
            pn.pane.HTML(mapping_hud, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 🔎 APSO Note (Pre-Redaction — Leakage Visible)",
                    pn.pane.HTML(
                        f'{ink_style}<div class="forensic-text">{apso_note}</div>'
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Source Dialogue",
                    pn.pane.Str(
                        dialogue,
                        styles={'white-space': 'pre-wrap', 'font-weight': '500'}
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                )
            )
        )

    layout = pn.Column(
        "# 🕵️ Leakage Forensic Auditor — Phase 3b.1",
        f"**Objective:** Visually confirm that the {len(df_pd):,} records "
        "identified by Phase 3b genuinely contain ICD-10 code strings "
        "before redaction is applied in Phase 3c. "
        "Look for patterns like `(ICD-10: M25.562)`, `(ICD-10 code N39.0)`, "
        "or bare codes like `M25.562` in the Assessment section.",
        index_slider,
        get_leakage_view,
        sizing_mode='stretch_width'
    )

    return layout.show(title="Leakage Forensic Auditor — Phase 3b.1", threaded=True)


# ==============================================================================
# LAUNCH AUDITOR
# Uses df_explicit_leaks produced by Phase 3b
# ==============================================================================
if 'df_explicit_leaks' not in locals() or df_explicit_leaks is None:
    print("❌ df_explicit_leaks not found. Please run Phase 3b first.")
else:
    leak_server = launch_leakage_auditor(df_explicit_leaks)
    print("\n💡 TIP: Navigate the slider to sample different leaking records.")
    print("   Confirm that code strings are genuinely present in the APSO note.")
    print("   When satisfied, proceed to Phase 3c for redaction.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔒 Phase 3c: ICD-10 Code Redaction

Having quantified the leakage in Phase 3b — 2,923 records (28.5%) containing
explicit ICD-10 code strings, concentrated in the Assessment section — we now
apply targeted redaction to eliminate the leakage before any downstream model
training occurs.

### Redaction Strategy

ICD-10 code strings are replaced with `[REDACTED]` rather than deleted
entirely. This preserves sentence structure and makes the redaction auditable:

> *"Diagnosis: Pain in left knee (ICD-10: M25.562)"*
> becomes:
> *"Diagnosis: Pain in left knee (ICD-10: [REDACTED])"*

A bare deletion would leave syntactically broken fragments like
*"Diagnosis: Pain in left knee (ICD-10: )"* which could confuse tokenisers.
The `[REDACTED]` marker signals to human reviewers exactly where codes were
removed without disrupting the surrounding clinical language.

### Scope

All four SOAP section columns are redacted (`assessment`, `plan`, `objective`,
`subjective`), then `apso_note` is rebuilt from the cleaned sections. This
ensures `apso_note` is the single source of truth for downstream model input —
fully cleaned, Assessment-first, zero code strings.

The original `note` column is **not modified**. It is preserved in the Gold
layer for audit traceability and for tasks where the original unredacted text
is needed.

### What Is and Isn't Changed

| Column | Action |
|---|---|
| `assessment`, `plan`, `objective`, `subjective` | ICD-10 strings replaced with `[REDACTED]` |
| `apso_note` | Rebuilt from redacted sections |
| `note` | **Preserved unchanged** — original SOAP text |
| `standard_icd10`, `label`, `code_status` | **Preserved unchanged** — labels intact |
| All other columns | **Preserved unchanged** |

No records are removed. The labels (`standard_icd10`, `label`) are not touched —
redaction targets only the text fields where code strings appear as leakage.

### Post-Redaction Verification

The same detection pattern from Phase 3b is re-applied to the cleaned
`apso_note` and all section columns after redaction. The expected result is
0 remaining leaks. Any residual leaks would indicate code formats not covered
by the pattern and would block progression to downstream training.

> **Result:** A fully redacted Gold layer where `apso_note` contains zero
> explicit ICD-10 code strings — forcing any downstream model to learn from
> clinical language rather than label pattern-matching.

</div>

In [ ]:
# ==============================================================================
# PHASE 3c: ICD-10 CODE REDACTION
# ==============================================================================
# Purpose: Remove explicit ICD-10 code strings from all SOAP section columns
# and rebuild the apso_note from the cleaned sections.
#
# Strategy: Replace ICD-10 code strings with [REDACTED] rather than deleting
# them entirely. This preserves sentence structure — for example:
#   "Diagnosis: Pain in left knee (ICD-10: M25.562)"
#   becomes:
#   "Diagnosis: Pain in left knee (ICD-10: [REDACTED])"
# This is more readable than a bare deletion and makes redaction auditable —
# the [REDACTED] marker signals to a human reviewer exactly where codes were
# removed without disrupting the surrounding clinical language.
#
# Scope: All four SOAP section columns are cleaned (assessment, plan,
# objective, subjective), then apso_note is rebuilt from the cleaned sections.
# The original 'note' column is NOT modified — it is preserved for audit
# traceability and for tasks where the original text is needed.
#
# This phase does not remove records. Every record in df_gold retains its
# full clinical content — only the explicit code strings are redacted.
# ==============================================================================

import polars as pl

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 3c Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    print("❌ CRITICAL: df_gold not found. Please run Phase 3b first.")
    raise ValueError("df_gold not available in local scope")

required_cols = {'apso_note', 'assessment', 'plan', 'subjective', 'objective'}
missing_cols  = required_cols - set(df_gold.columns)
if missing_cols:
    raise ValueError(
        f"❌ CRITICAL: df_gold is missing required columns: {missing_cols}"
    )

# Confirm Phase 3b leakage counts are available for post-redaction verification
if 'explicit_leak_count' not in locals():
    print("   ⚠️  explicit_leak_count not found in scope.")
    print("      Post-redaction verification will compute baseline from scratch.")
    explicit_leak_count = None

print(f"   ✅ df_gold available: {len(df_gold):,} records")
print(f"   ✅ All required SOAP columns present")

# ==============================================================================
# 2. DEFINE REDACTION PATTERN AND REPLACEMENT
# ==============================================================================
# Same pattern as Phase 3b — anchored on letter + two digits to discriminate
# ICD-10 codes from medical abbreviations. Using the same pattern guarantees
# that Phase 3b's detection counts and Phase 3c's redaction counts are
# directly comparable.
# ==============================================================================
ICD10_REDACT_PATTERN = (
    r'\b[A-Z][0-9]{2}\.[0-9A-Z]{1,4}\b'   # canonical: M25.562, N39.0
    r'|'
    r'\b[A-Z][0-9]{2}[0-9A-Z]{0,5}\b'     # raw: M25562, N390, J18
)
REDACTION_MARKER = "[REDACTED]"

print(f"\n🔧 Redaction configuration:")
print(f"   Pattern:     {ICD10_REDACT_PATTERN}")
print(f"   Replacement: {REDACTION_MARKER}")

# ==============================================================================
# 3. REDACT SOAP SECTION COLUMNS
# ==============================================================================
# Apply redaction to each SOAP section column independently.
# Null values are passed through unchanged (str.replace on null returns null).
# ==============================================================================
print("\n🔄 Applying redaction to SOAP section columns...")

SECTIONS_TO_REDACT = ['assessment', 'plan', 'objective', 'subjective']

# Build all redaction expressions in a single with_columns call for efficiency
redaction_exprs = [
    pl.col(section)
    .str.replace_all(ICD10_REDACT_PATTERN, REDACTION_MARKER)
    .alias(f"{section}_clean")
    for section in SECTIONS_TO_REDACT
]

df_gold = df_gold.with_columns(redaction_exprs)

# Report how many replacements were made per section
print(f"\n   📊 Redaction counts per section:")
for section in SECTIONS_TO_REDACT:
    # Count records where the cleaned version differs from the original
    changed = df_gold.filter(
        pl.col(section).is_not_null() &
        (pl.col(f"{section}_clean") != pl.col(section))
    ).height
    pct = changed / len(df_gold) * 100
    print(f"   ✅ {section.capitalize():12s}: {changed:,} records redacted ({pct:.1f}%)")

# ==============================================================================
# 4. REPLACE ORIGINAL SECTION COLUMNS WITH CLEANED VERSIONS
# ==============================================================================
# Overwrite the original section columns with the cleaned versions,
# then drop the temporary _clean columns.
# ==============================================================================
print("\n🔄 Replacing original section columns with redacted versions...")

replace_exprs = [
    pl.col(f"{section}_clean").alias(section)
    for section in SECTIONS_TO_REDACT
]

df_gold = df_gold.with_columns(replace_exprs).drop(
    [f"{section}_clean" for section in SECTIONS_TO_REDACT]
)

print(f"   ✅ Original section columns replaced with redacted versions")

# ==============================================================================
# 5. REBUILD APSO NOTE FROM CLEANED SECTIONS
# ==============================================================================
# The apso_note column was built from the original (unredacted) section columns
# in Phase 3a. Now that the section columns are cleaned, we rebuild apso_note
# to reflect the redacted content.
#
# This ensures apso_note is the single source of truth for downstream
# model input — fully cleaned, Assessment-first, no code strings.
# ==============================================================================
print("\n🔄 Rebuilding apso_note from cleaned sections...")

df_gold = df_gold.with_columns([
    (
        pl.col("assessment").fill_null("") + "\n\n" +
        pl.col("plan").fill_null("")       + "\n\n" +
        pl.col("subjective").fill_null("") + "\n\n" +
        pl.col("objective").fill_null("")
    ).str.strip_chars().alias("apso_note")
])

print(f"   ✅ apso_note rebuilt from redacted sections")

# ==============================================================================
# 6. POST-REDACTION VERIFICATION
# ==============================================================================
# Re-run the Phase 3b detection pattern on the cleaned apso_note to confirm
# that redaction drove the leak count to 0.
# ==============================================================================
print("\n🔍 Post-redaction verification...")

remaining_leaks = df_gold.filter(
    pl.col("apso_note").str.contains(ICD10_REDACT_PATTERN)
).height

remaining_by_section = {}
for section in SECTIONS_TO_REDACT:
    remaining = df_gold.filter(
        pl.col(section).is_not_null() &
        pl.col(section).str.contains(ICD10_REDACT_PATTERN)
    ).height
    remaining_by_section[section] = remaining

print(f"\n📊 POST-REDACTION LEAK CHECK:")
print(f"   " + "─" * 50)

if explicit_leak_count is not None:
    print(f"   📊 Pre-redaction leaks (apso_note):  {explicit_leak_count:,}")

icon = "✅" if remaining_leaks == 0 else "❌"
print(f"   {icon} Post-redaction leaks (apso_note): {remaining_leaks:,}")

all_clean = True
for section, remaining in remaining_by_section.items():
    icon = "✅" if remaining == 0 else "❌"
    if remaining > 0:
        all_clean = False
    print(f"   {icon} {section.capitalize():12s}: {remaining:,} remaining leaks")

print(f"   " + "─" * 50)

if all_clean and remaining_leaks == 0:
    print(f"   ✅ REDACTION COMPLETE: 0 ICD-10 code strings remain")
else:
    print(f"   ❌ WARNING: {remaining_leaks:,} leaks remain after redaction")
    print(f"      Review pattern coverage — some code formats may not be captured")

# ==============================================================================
# 7. APSO NOTE PREVIEW (POST-REDACTION)
# ==============================================================================
print(f"\n👁️  APSO Note Preview (First 3 Records — post-redaction):")

# Show the same records as Phase 3b sample to allow direct comparison
preview_ids = ["0", "1", "2"]
for record_id in preview_ids:
    row = df_gold.filter(pl.col("id") == record_id)
    if row.height > 0:
        apso_preview = (row["apso_note"][0] or "")[:300].replace("\n", " ↵ ")
        code = row["standard_icd10"][0]
        print(f"\n   ID: {record_id} | Code: {code}")
        print(f"   {apso_preview}...")

# ==============================================================================
# 8. REGISTER UPDATED GOLD LAYER
# ==============================================================================
print(f"\n💾 Re-registering redacted Gold layer...")
config.register_dataframe("gold_medsynth", df_gold, phase="Phase 3c")
print(f"   ✅ gold_medsynth updated: {len(df_gold):,} records, "
      f"{len(df_gold.columns)} columns")
print(f"   ✅ Original 'note' column preserved (unredacted) for audit traceability")

# ==============================================================================
# 9. AUDIT TRAIL
# ==============================================================================
config.log_event(
    phase="Phase 3c: ICD-10 Redaction",
    action="redaction_complete",
    details={
        "total_records":              len(df_gold),
        "pre_redaction_leaks":        explicit_leak_count if explicit_leak_count else "unknown",
        "post_redaction_leaks":       remaining_leaks,
        "redaction_successful":       remaining_leaks == 0,
        "redaction_marker":           REDACTION_MARKER,
        "sections_redacted":          SECTIONS_TO_REDACT,
        "apso_note_rebuilt":          True,
        "original_note_preserved":    True,
        "section_remaining_leaks":    remaining_by_section
    }
)

print(f"\n📝 Audit trail updated")

# ==============================================================================
# 10. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")

if remaining_leaks == 0:
    print(f"   ✅ 0 ICD-10 code strings remain in apso_note")
    print(f"   ✅ 0 ICD-10 code strings remain in any SOAP section column")
    print(f"   ✅ apso_note rebuilt from redacted sections")
    print(f"   ✅ Original SOAP note preserved in 'note' column")
    print(f"   ✅ Gold layer ready for downstream model training")
else:
    print(f"   ❌ {remaining_leaks:,} ICD-10 code strings remain")
    print(f"   ❌ Review redaction pattern and re-run before proceeding")

print(f"\n✅ PHASE 3c COMPLETE: ICD-10 codes redacted from all SOAP sections")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📖 Interpretation of Results

This section synthesises the findings from the full ingestion, validation, and
preparation pipeline. It is intended as a standing reference before any
downstream modelling phase begins.

---

### 1. The Dataset

MedSynth is a fully synthetic dataset of 10,240 doctor-patient dialogue and
clinical note pairs, generated by a four-agent GPT-4o pipeline informed by
real-world disease distributions from 800 million IQVIA insurance claims.
Notes follow strict SOAP format; dialogues are derived from notes by a
separate generation pipeline.

The dataset was engineered with **uniform sampling** — exactly 5 records per
ICD-10 code — deliberately preventing domination by common diseases. This
produces a perfectly balanced label distribution across 2,037 distinct codes,
with a minimum frequency of 5 per code and zero rare labels. This is an unusual
property that should be explicitly acknowledged in any evaluation design:
models trained on this data will not learn to handle real-world class imbalance.

---

### 2. ICD-10 Label Quality

Validation against the CDC FY2026 reference confirmed the following
classification across all 10,240 records:

| Status | Records | % | Interpretation |
|---|---|---|---|
| `billable` | 9,660 | 94.3% | Leaf-level codes confirmed in CDC tabular reference |
| `noisy_111` | 555 | 5.4% | Valid parent codes reflecting real billing practice |
| `placeholder_x` | 25 | 0.24% | Legitimate injury codes absent from CM descriptions file |

The 5.4% Noisy 111 rate is not a data quality problem. The MedSynth authors
selected from the top 2,001 most frequent ICD-10 codes in real insurance
claims, and clinicians genuinely submit category-level codes in practice.
These records are retained in the Gold layer with a `code_status` annotation.

The 25 placeholder-X codes (e.g. `T78.1XXA`) are structurally valid ICD-10-CM
injury codes using the mandatory placeholder X convention. Their absence from
the CDC descriptions file is a known gap in the tabular reference, not a
dataset error.

---

### 3. Token Pressure

Both text fields exceed ClinicalBERT's 512-token context window on average,
consistent with the MedSynth paper's own reported statistics:

| Field | Paper avg (tokens) | Records exceeding 512 |
|---|---|---|
| Note | 621 | 64.7% |
| Dialogue | 932 | 100.0% |

The APSO-Flip transformation (Phase 3a) addresses this by reordering the note
to place the Assessment section — which contains the primary diagnostic signal
— at Token 0. After the flip, 64.8% of notes still exceed the window, but the
content that survives truncation is now the diagnostically critical content
rather than the Subjective section preamble.

The `words × 1.3` token heuristic used in Phase 1c is conservative for
ClinicalBERT specifically, which uses WordPiece tokenisation and tends to
produce more tokens per word than the BPE tokenisers used to report counts in
the MedSynth paper. True truncation rates for ClinicalBERT are likely higher
than reported here.

---

### 4. SOAP Extraction

The Phase 1e Pydantic Gatekeeper achieved 100% extraction success across all
four SOAP sections. This result is expected for a synthetically generated
dataset — the MedSynth Note Writer Agent was explicitly instructed to produce
SOAP-formatted notes, and the Note Polisher Agent enforced correct section
placement. This extraction rate should not be assumed to generalise to real
EHR notes.

The Subjective section varies stochastically across records: some include
CC/HPI/ROS sub-sections, others do not. This is by design — the Note Writer
Agent prompt instructed the model to "roll a dice" to determine sub-section
inclusion.

---

### 5. ICD-10 Code Leakage

The MedSynth Note Writer Agent embedded ICD-10 code strings directly in the
Assessment section of 27.9% of records (2,855 records). This constitutes
data leakage for any ICD-10 classification task — a model trained on unredacted
notes can predict labels by pattern-matching the code string rather than
reasoning from clinical language.

Phase 3b quantified the leakage precisely:

| Section | Leaking records | % |
|---|---|---|
| Assessment | 2,855 | 27.9% |
| Plan | 238 | 2.3% |
| Objective | 89 | 0.9% |
| Subjective | 60 | 0.6% |
| **Total unique** | **2,923** | **28.5%** |

Phase 3c redacted all explicit code strings using a pattern anchored on the
ICD-10 structural convention (letter + two digits), replacing them with
`[REDACTED]` to preserve sentence structure. Post-redaction verification
confirmed 0 remaining leaks across all sections.

The original `note` column is preserved unredacted for tasks where the
original text is required.

---

### 6. What the Gold Layer Provides

The final Gold layer (`df_gold` / `gold_medsynth`) contains 10,240 records
and 13 columns, providing multiple representations of each clinical encounter:

* **`note`** — original SOAP note, unmodified, for reference and non-classification tasks
* **`apso_note`** — APSO-recomposed, redacted note for classification training
* **`standard_icd10`** — canonical decimal-restored ICD-10 code (e.g. `M25.562`)
* **`code_status`** — CDC-grounded classification (`billable`, `noisy_111`, `placeholder_x`)
* **`assessment`, `plan`, `subjective`, `objective`** — extracted, redacted SOAP sections

---

### 7. Recommended Downstream Configurations

Downstream phases should select records and text fields based on their specific
modelling objective:
```python

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🛡️ Phase 3c.1: Post-Redaction Integrity Auditor

Following Phase 3c's automated verification of 0 remaining leaks, this
browser-native auditor provides human confirmation that the redaction is
visually correct before the Gold layer is committed for downstream use.

### What to Confirm

Navigate across a representative sample of records and verify:

1. **Redaction markers are present** — `[REDACTED]` should appear where ICD-10
   code strings were. The HUD shows how many redactions fired for each record.
   Records with 0 redactions had no code strings to begin with.

2. **Clinical language is intact** — the text surrounding each `[REDACTED]`
   marker should read naturally (e.g. *"Pain in the left knee (ICD-10: [REDACTED])"*).
   The clinical description of the diagnosis should still be present.

3. **APSO ordering is preserved** — Assessment content should appear at the
   top of the note (green panel), followed by Plan, then Subjective, then
   Objective.

4. **No visible code strings remain** — scan the full note for any `X##.###`
   or `X##X##` patterns that may have been missed.

Sample across `billable`, `noisy_111`, and `placeholder_x` records using the
`code_status` shown in the HUD. The soft green background of the note panel
distinguishes this post-redaction view from the pre-redaction amber view in
Phase 3b.1.

> **Final Gate:** Once satisfied, the Gold layer is ready for downstream model
> training. Close the auditor before proceeding.

</div>

In [ ]:
# ==============================================================================
# PHASE 3c.1: POST-REDACTION INTEGRITY AUDITOR
# ==============================================================================
# Purpose: Visual confirmation that Phase 3c redaction is correct on a
# record-by-record basis. Complements the automated 0-leak verification
# by allowing human inspection of the redacted apso_note alongside the
# original dialogue.
#
# What to confirm:
#   1. [REDACTED] markers appear where ICD-10 codes were
#   2. Surrounding clinical language is intact and readable
#   3. Assessment section appears at the top (APSO ordering preserved)
#   4. No code strings are visually detectable in the note
# ==============================================================================

import panel as pn
pn.extension(design='material')


def launch_sanitization_auditor(df_redacted):
    df_pd = df_redacted.to_pandas()

    index_slider = pn.widgets.IntSlider(
        name='Verify Redacted Record (Index)',
        start=0, end=len(df_pd) - 1, value=0,
        bar_color='#27ae60', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def get_audit_view(index):
        row           = df_pd.iloc[index]
        standard_label = row.get('standard_icd10',      'N/A')
        code_status    = row.get('code_status',          'N/A')
        apso_note      = row.get('apso_note',            '⚠️ apso_note not found.')
        dialogue       = row.get('dialogue',             '⚠️ dialogue not found.')
        token_est      = row.get('apso_token_estimate',  0)

        # Check for any remaining [REDACTED] markers — confirms redaction fired
        redacted_count = apso_note.count('[REDACTED]') if isinstance(apso_note, str) else 0
        redact_label   = (f"✅ {redacted_count} code(s) redacted"
                          if redacted_count > 0
                          else "ℹ️  No codes present in this record")

        mapping_hud = f"""
        <div style="display: flex; gap: 20px; max-width: 850px;
                    margin-bottom: 20px; font-family: sans-serif;">
            <div style="flex: 1; padding: 15px; background: #fff;
                        border: 2px solid #27ae60; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.8rem; color: #27ae60; font-weight: bold;
                             text-transform: uppercase;">Gold Label</span><br>
                <span style="font-size: 1.8rem; color: #27ae60;
                             font-weight: 900;">{standard_label}</span><br>
                <span style="font-size: 0.8rem; color: #666;">{code_status}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: #2c3e50;
                        border-radius: 8px; text-align: center; color: white;">
                <span style="font-size: 0.8rem; font-weight: bold;
                             text-transform: uppercase;">Redaction Status</span><br>
                <span style="font-size: 1.0rem; font-weight: 700;">
                    🛡️ ZERO-LEAK VERIFIED</span><br>
                <span style="font-size: 0.85rem; margin-top: 4px;
                             display: block;">{redact_label}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: #f8f9fa;
                        border: 2px solid #ddd; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.8rem; color: #666; font-weight: bold;
                             text-transform: uppercase;">Est. Tokens</span><br>
                <span style="font-size: 1.8rem; color: #333;
                             font-weight: 900;">{int(token_est):,}</span>
            </div>
        </div>
        """

        ink_style = """
        <style>
            .sanitized-text {
                white-space: pre-wrap !important;
                font-family: 'Courier New', monospace !important;
                font-size: 1.0rem !important;
                font-weight: 700 !important;
                color: #000 !important;
                background-color: #e8f8f5 !important;
                padding: 20px;
                border: 2px solid #27ae60;
                line-height: 1.5;
            }
        </style>
        """

        return pn.Column(
            pn.pane.HTML(mapping_hud, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 🛡️ Redacted APSO Note (Training View)",
                    pn.pane.HTML(
                        f'{ink_style}<div class="sanitized-text">{apso_note}</div>'
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Original Dialogue (Ground Truth)",
                    pn.pane.Str(
                        dialogue,
                        styles={'white-space': 'pre-wrap', 'font-weight': '500'}
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                )
            )
        )

    layout = pn.Column(
        "# 🛡️ Post-Redaction Integrity Auditor — Phase 3c.1",
        f"**Objective:** Visually confirm that [REDACTED] markers appear correctly "
        f"and no ICD-10 code strings remain visible. Check that Assessment content "
        f"appears first (APSO ordering) and surrounding clinical language is intact. "
        f"Showing {len(df_pd):,} records.",
        index_slider,
        get_audit_view,
        sizing_mode='stretch_width'
    )

    return layout.show(title="Post-Redaction Integrity Auditor — Phase 3c.1",
                       threaded=True)


# ==============================================================================
# LAUNCH AUDITOR
# Uses df_gold produced by Phase 3c
# ==============================================================================
if 'df_gold' not in locals() or df_gold is None:
    print("❌ df_gold not found. Please run Phase 3c first.")
else:
    print("🚀 Launching Post-Redaction Integrity Auditor (Phase 3c.1)...")
    print("   💡 TIP: Navigate the slider to sample redacted records.")
    print("   💡 TIP: Look for [REDACTED] markers in the Assessment section.")
    print("   💡 TIP: Confirm surrounding clinical language is intact.")
    print("   💡 TIP: Sample across billable, noisy_111, and placeholder_x records.")
    final_audit_server = launch_sanitization_auditor(df_gold)

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 💾 Phase 4: Gold Layer Parquet Export

This is the final checkpoint of the preprocessing pipeline. We persist the
fully prepared Gold layer to a Parquet file, transitioning from an active
transformation pipeline to a fixed, versioned artifact ready for downstream
model training or analysis.

### Export Strategy

1. **Parquet Format:** Parquet preserves the strict Polars schema — string
   `id`, `List(String)` labels, `Float64` token estimates — ensuring type
   consistency when the artifact is loaded in a downstream notebook without
   needing to re-run the full preprocessing pipeline.

2. **Snappy Compression:** Balances disk footprint with fast read/write
   performance, appropriate for iterative training workflows.

3. **Immutable Artifact:** This export freezes all transformations applied
   throughout the pipeline — APSO reordering, decimal restoration, status
   annotation, and ICD-10 redaction. The `apso_note` and `standard_icd10`
   columns in this file are the canonical training inputs.

4. **Audit Closure:** The export event is logged to the audit trail with
   full metadata — path, row count, column set, and pipeline completion
   timestamp — closing the preprocessing audit record.

> **Result:** A versioned, schema-consistent Parquet artifact containing all
> 10,240 annotated Gold layer records, ready for downstream use without
> re-running any preprocessing step.

</div>

In [ ]:
# ==============================================================================
# PHASE 4: GOLD LAYER PARQUET EXPORT
# ==============================================================================

import polars as pl
from datetime import datetime

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 4 Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    print("❌ CRITICAL: df_gold not found. Please run Phase 3c first.")
    raise ValueError("df_gold not available in local scope")

print(f"   ✅ df_gold available: {len(df_gold):,} records, {len(df_gold.columns)} columns")

# Confirm redaction is complete before exporting
icd10_check = df_gold.filter(
    pl.col("apso_note").str.contains(
        r'\b[A-Z][0-9]{2}\.[0-9A-Z]{1,4}\b|\b[A-Z][0-9]{2}[0-9A-Z]{0,5}\b'
    )
).height

if icd10_check > 0:
    print(f"   ❌ WARNING: {icd10_check:,} records still contain ICD-10 code strings.")
    print(f"      Please run Phase 3c before exporting.")
    raise ValueError("Redaction incomplete — do not export until Phase 3c is complete.")
else:
    print(f"   ✅ Redaction verified: 0 ICD-10 code strings in apso_note")

# ==============================================================================
# 2. DEFINE EXPORT PATH
# ==============================================================================
gold_dir    = config.resolve_path("data", "gold")
timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
export_path = gold_dir / f"medsynth_gold_apso_{timestamp}.parquet"

print(f"\n💾 Export path: {export_path}")

# ==============================================================================
# 3. WRITE PARQUET
# ==============================================================================
df_gold.write_parquet(export_path, compression="snappy")

file_size_mb = export_path.stat().st_size / (1024 * 1024)
print(f"   ✅ Parquet written: {file_size_mb:.1f} MB")

# ==============================================================================
# 4. VERIFY THE WRITTEN FILE
# ==============================================================================
# Load back and confirm row count and schema match
df_verify = pl.read_parquet(export_path)

if df_verify.height != len(df_gold) or df_verify.columns != df_gold.columns:
    print(f"   ❌ WARNING: Written file does not match source DataFrame.")
    print(f"      Source: {len(df_gold):,} rows, {df_gold.columns}")
    print(f"      Written: {df_verify.height:,} rows, {df_verify.columns}")
    raise ValueError("Parquet export verification failed.")
else:
    print(f"   ✅ Verification passed: {df_verify.height:,} rows, "
          f"{len(df_verify.columns)} columns")

# ==============================================================================
# 5. AUDIT TRAIL
# ==============================================================================
config.log_event(
    phase="Phase 4: Parquet Export",
    action="gold_parquet_exported",
    details={
        "export_path":        str(export_path),
        "file_size_mb":       round(file_size_mb, 2),
        "total_rows":         len(df_gold),
        "total_columns":      len(df_gold.columns),
        "columns":            df_gold.columns,
        "billable_count":     df_gold.filter(pl.col("code_status") == "billable").height,
        "noisy_111_count":    df_gold.filter(pl.col("code_status") == "noisy_111").height,
        "placeholder_x_count": df_gold.filter(pl.col("code_status") == "placeholder_x").height,
        "icd10_leaks_verified": 0,
        "compression":        "snappy",
        "primary_text_col":   "apso_note",
        "label_col":          "standard_icd10",
        "status_col":         "code_status"
    }
)

print(f"\n📝 Audit trail updated — preprocessing pipeline closed")

# ==============================================================================
# 6. COMPLETION SUMMARY
# ==============================================================================
print(f"\n🚀 PIPELINE COMPLETE")
print(f"   " + "─" * 50)
print(f"   📊 Records exported:      {len(df_gold):,}")
print(f"   💾 File size:             {file_size_mb:.1f} MB")
print(f"   📁 Location:              {export_path.name}")
print(f"   ✅ Billable:              "
      f"{df_gold.filter(pl.col('code_status') == 'billable').height:,}")
print(f"   ℹ️  Noisy 111:            "
      f"{df_gold.filter(pl.col('code_status') == 'noisy_111').height:,}")
print(f"   🔍 Placeholder-X:         "
      f"{df_gold.filter(pl.col('code_status') == 'placeholder_x').height:,}")
print(f"   🔒 ICD-10 leaks:          0")
print(f"   🔄 APSO-flipped:          Yes")
print(f"   " + "─" * 50)
print(f"   📌 Primary input column:  apso_note")
print(f"   📌 Label column:          standard_icd10")
print(f"   📌 Status column:         code_status")
print(f"   📌 Original note:         note (unredacted)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🏁 Pipeline Conclusion

This notebook has taken the MedSynth dataset from raw ingestion through to a
fully validated, annotated, and redacted Gold layer — ready for downstream
model training or analysis.

### Final Statistics

| Metric | Value | Notes |
|---|---|---|
| Total records | 10,240 | Full dataset retained — no records removed |
| Billable codes | 9,660 (94.3%) | Confirmed leaf codes, CDC FY2026 reference |
| Noisy 111 codes | 555 (5.4%) | Valid parent codes — real billing practice |
| Placeholder-X codes | 25 (0.24%) | Legitimate injury codes, retained with status |
| APSO coverage | 100% | Assessment at Token 0 for every record |
| ICD-10 leaks remaining | 0 | Verified post-redaction across all columns |
| Export size | 60.6 MB | Snappy-compressed Parquet |

### What Was Built

Starting from 10,240 raw synthetic clinical records, the pipeline produced a
Gold layer (`medsynth_gold_apso_20260319_111429.parquet`) with 13 columns
providing multiple representations of each encounter:

* **`apso_note`** — APSO-recomposed, ICD-10-redacted note for classification training
* **`note`** — original SOAP note, unmodified, for reference tasks
* **`standard_icd10`** — canonical decimal-restored ICD-10 code
* **`code_status`** — CDC-grounded classification (`billable`, `noisy_111`, `placeholder_x`)
* **`assessment`, `plan`, `subjective`, `objective`** — extracted, redacted SOAP sections
* **`label`** — raw ICD-10 label array, preserved from source
* **`dialogue`** — original doctor-patient transcript, unmodified

### Important Caveats for Downstream Use

1. **Class balance is artificial.** MedSynth was engineered with uniform
   sampling (5 records per code). Models trained on this data will not learn
   real-world class frequency distributions. Evaluation on held-out MedSynth
   splits will not reflect deployment performance on real clinical data.

2. **`[REDACTED]` is a learnable token.** The redaction marker replaces code
   strings but is itself present in 28.5% of records. Models with sufficient
   capacity may learn to associate `[REDACTED]` with specific codes via
   surrounding context. Final evaluation should ideally use a real clinical
   dataset such as MIMIC-III.

3. **Dialogue leakage has not been assessed.** The `dialogue` column has not
   been scanned or redacted. The Dialogue Polisher Agent was designed to
   ensure all note content — including ICD-10 codes — appears in the dialogue.
   If dialogue is used as model input, a separate leakage detection pass is
   required before training.

4. **Markdown formatting is present.** The `apso_note` column contains `**`
   bold markers from the Note Writer Agent's output. These consume context
   window tokens without adding clinical signal. Strip markdown formatting
   before tokenisation.

---

**Next Step:** Proceed to Notebook 02 for model training. Use `apso_note` as
the primary text input and `standard_icd10` as the label. Filter by
`code_status` as appropriate for your modelling objective.

</div>